## KV Cache Optimisation — Ideas 1, 2 & 4 (Diagonal K/V)

Three ideas for reducing KV cache memory at inference time:

**Idea 1 — XSA (Exclusive Self-Attention):** Removes attention-sink bias from outputs,
making PageRank eviction scores semantically meaningful.

**Idea 2 — Keyword KV Cache (rustworkx PageRank):** Evicts low-importance tokens from
the old context using PageRank on the attention graph. 66% memory reduction, zero PPL loss.

**Idea 4 — Diagonal W_K / W_V with Exact V Recovery:**
Restrict `W_K = diag(w_k)` and `W_V = diag(w_v)` — element-wise scaling only.
Then `K = x * w_k`, `V = x * w_v`, so `V = K * (w_v / w_k)` exactly.
The ratio vector `r = w_v / w_k` (d_head scalars, shared across all tokens) is stored
once per head. At inference only K is cached; V is reconstructed on the fly: `V = K * r`.
This gives **exact 50% KV memory reduction with zero approximation error**.

**Tradeoff:** W_K and W_V lose rotational expressivity (O(d) params vs O(d²)).
W_Q stays full-rank — it is not cached so there is no reason to restrict it.
The model must be trained from scratch; pretrained GPT-2 weights cannot be used.
Heads specialise via their diagonal scaling patterns rather than arbitrary subspace projections.

**Idea 3 (V Deduplication):** Retired — fundamentally incompatible with GPT-2 absolute
position embeddings. Works only on RoPE models (LLaMA/Mistral) where V is position-free.


In [32]:
import math, urllib.request, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from transformers import GPT2Model
import rustworkx as rx           # ← Rust-backed graph library for PageRank

warnings.filterwarnings('ignore')
torch.manual_seed(42)

MAX_TOKENS       = 4096
KEYWORD_RATIO    = 0.3   # Idea 2: keep top 30% of tokens after threshold
KEYWORD_WINDOW   = 256   # Idea 2: tokens before this = full cache; after = keyword only
DEDUP_ENABLED    = True  # Idea 3: deduplicate V by token id
EIGENBASIS_RANK  = 32    # Idea 4: rank-r approx of shared eigenbasis (None = disabled)

In [33]:
# ── Shared layers (unchanged from original) ───────────────────────────────────

class LayerNorm(nn.Module):
    """Fixed: unbiased=False variance matches GPT-2 paper."""
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta  = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        return (x - mean) / (var + self.eps).sqrt() * self.gamma + self.beta

class GeLU(nn.Module):
    def forward(self, x): return F.gelu(x)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['emb_dim']
        self.layers = nn.Sequential(nn.Linear(d, 4*d), GeLU(), nn.Linear(4*d, d))
    def forward(self, x): return self.layers(x)


In [34]:
# ── Baseline: standard softmax attention (original notebook) ──────────────────

class CausalAttentionSoftmax(nn.Module):
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        return self.drop(torch.softmax(s, dim=-1)) @ v

class MHASoftmax(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionSoftmax(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)
    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))


## Idea 1 — Exclusive Self-Attention (XSA)

**Paper:** Zhai, Apple 2026 — arXiv:2603.09078

**Problem it solves:** Standard attention output `y_i` has high cosine similarity
with the token's own value vector `v_i`. This means SA wastes capacity doing
pointwise feature transforms that FFN should handle, and creates artificial
**attention sinks** which corrupt KV eviction importance scores.

**Mechanism** — one extra step after attention:
```
y_i  = Σ a_ij · v_j          # standard weighted sum
z_i  = y_i - (y_i · v̂_i) · v̂_i   # remove self-value component
```
Projects out the component of `y_i` along `v_i`. Forces attention to carry
only **new contextual information** orthogonal to what the token already knows.

**KV cache impact:** No size reduction directly. But by eliminating attention sinks,
Idea 2's eviction scores become semantically meaningful — tokens that matter
get high scores, tokens that don't get evicted cleanly.


**Perplexity note:** XSA must be trained from scratch to benefit PPL — applying the
projection post-hoc to GPT-2 pretrained weights removes signal the model relies on.
We use `apply_xsa=False` during PPL measurement and demonstrate the projection
mechanics separately in the attention-sink analysis section.


In [35]:
# ── Idea 1: XSA — Exclusive Self-Attention ────────────────────────────────────
# Zhai, Apple 2026 (arXiv:2603.09078)
#
# IMPORTANT: The XSA projection is learned behaviour — it must be trained in
# from scratch.  Applying it post-hoc to GPT-2 pretrained weights removes signal
# the model already learnt to rely on, so PPL rises.
# We therefore use apply_xsa=False for perplexity measurement and apply_xsa=True
# only for the attention-sink analysis where we demonstrate the projection works.

class CausalAttentionXSA(nn.Module):
    """
    Standard causal attention + XSA projection step.
    After computing y = softmax(QK^T/√d)·V, we project out the
    component of y along the token's own value vector v_i:

        z_i = y_i - (y_i · v̂_i) · v̂_i

    This eliminates attention similarity bias / attention sinks.
    Use apply_xsa=False at inference on pretrained weights (PPL measurement).
    Use apply_xsa=True only when the model was trained with XSA from scratch.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.apply_xsa = False   # ← set True only when trained from scratch

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = self.drop(torch.softmax(s, dim=-1))
        y = attn @ v

        if self.apply_xsa:
            # Remove self-value projection: z_i = y_i - (y_i·v̂_i)·v̂_i
            v_norm = F.normalize(v, dim=-1)
            proj   = (y * v_norm).sum(dim=-1, keepdim=True)
            y      = y - proj * v_norm

        return y

class MHAXSA(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionXSA(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def set_apply_xsa(self, flag: bool):
        for h in self.heads:
            h.apply_xsa = flag

    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]")


Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]


## Idea 2 — Keyword Threshold KV Cache (rustworkx PageRank)

**Idea:** Split the KV cache into two zones:
- **Recent window** (last `KEYWORD_WINDOW` tokens): keep full K and V — exact attention
- **Old context** (everything before): evict low-importance tokens, keep only top-k% as "keywords"

**How importance is computed — PageRank on the attention graph:**

When the recent zone flushes, we have `window/2` tokens and their pairwise attention
weights. We build a **directed weighted graph** in rustworkx:
- **Node** = one token position
- **Edge** `i → j` with weight `a_ij` = token `i` attends to token `j`

We then run `rustworkx.pagerank(G, alpha=0.85, weight_fn=...)`.

**Why PageRank?** A token that many other tokens attend to heavily will have many
high-weight incoming edges — exactly the tokens we want to keep.
PageRank propagates this signal transitively: if an important token attends to token X,
X also gets promoted. This is strictly better than the naive "mean attention received"
score because it captures the *network* structure of relevance, not just the raw degree.

**Why XSA makes this better:** Without XSA, the first token is always an artificial
attention sink with huge weight — it would dominate PageRank and never be evicted.
XSA removes this bias so the graph reflects true semantic importance.

**KV memory:**
```
Standard:   n × 2d  (grows forever)
Keyword:    (KEYWORD_WINDOW + k) × 2d   where k = KEYWORD_RATIO × old_context
            ≈ 30-60% reduction for long contexts
```

In [36]:
# ── Idea 2: Keyword Threshold KV Cache — rustworkx PageRank ──────────────────
#
# KEY INSIGHT: Build a directed attention graph for the flush batch using
# rustworkx.PyDiGraph.  Edge i→j carries weight = attention_weight[i,j].
# Run rustworkx.pagerank() — tokens that many others attend to get high score.
# Keep top KEYWORD_RATIO fraction by PageRank score; discard the rest.
#
# rustworkx is a Rust-backed Python graph library (used in Qiskit).
# It is significantly faster than networkx for this workload.

import rustworkx as rx

class KeywordKVCache:
    """
    Manages a two-zone KV cache with rustworkx PageRank importance scoring.

      Zone A (recent): last `window` tokens → full exact K, V stored.
      Zone B (old):    tokens beyond window → only top-k% 'keywords' kept.

    Importance scoring (flush time):
      1. Build PyDiGraph: node per token, edge i→j weighted by attn_weight[i,j].
      2. Run rustworkx.pagerank(G, alpha=0.85, weight_fn=lambda e: e).
      3. Rank nodes by PageRank score; keep top KEYWORD_RATIO fraction.

    Usage:
      cache = KeywordKVCache(window=256, ratio=0.3, d_head=64, device=device)
      cache.update(k_vec, v_vec, attn_row)   # one token at a time
      K, V  = cache.get_kv()                 # retrieve full usable KV tensors
      stats = cache.memory_stats()
    """

    def __init__(self, window: int, ratio: float, d_head: int, device):
        self.window  = window
        self.ratio   = ratio
        self.d_head  = d_head
        self.device  = device

        # Zone A — recent tokens (full K, V stored)
        self.recent_K    = []   # list of [d_head] CPU tensors
        self.recent_V    = []
        # attn_rows[t] = the attention ROW for token t:
        #   attn_rows[t][j] = weight that token t placed on token j (j ≤ t).
        # This is the row of the attention matrix, NOT the column.
        self.recent_attn_rows = []   # list of 1-D tensors, variable length

        # Zone B — evicted keyword tokens
        self.keyword_K = []
        self.keyword_V = []

        self.total_tokens       = 0
        self.pagerank_calls     = 0
        self.last_pr_scores     = None   # PageRank scores from last flush (for inspection)

    # ------------------------------------------------------------------
    def update(self, k: torch.Tensor, v: torch.Tensor,
               attn_row: torch.Tensor = None):
        """
        Add one token's K and V to the cache.

        attn_row: 1-D tensor of length (t+1) giving the attention weights
                  that THIS token placed on tokens 0..t-1 (i.e. row t of the
                  causal attention matrix: attn[t, 0..t]).
                  If None, a uniform weight of 1/n is assumed (degrades to
                  uniform PageRank, still correct).
        """
        self.recent_K.append(k.detach().cpu())
        self.recent_V.append(v.detach().cpu())

        n = len(self.recent_K)
        if attn_row is not None:
            row = attn_row.detach().cpu()
        else:
            row = torch.full((n,), 1.0 / n)

        self.recent_attn_rows.append(row)
        self.total_tokens += 1

        if len(self.recent_K) > self.window:
            self._flush_with_pagerank()

    # ------------------------------------------------------------------
    def _build_attention_graph(self, attn_rows, n):
        """
        Build a rustworkx PyDiGraph from the causal attention matrix.

        attn_rows[src] is the ATTENTION ROW for token `src`:
          attn_rows[src][dst] = weight that token `src` placed on token `dst`.
        Causal constraint: dst <= src (future tokens are not attended to).

        Edge direction: src → dst  ("token src attends to token dst")
        Weight = attention_weight[src, dst].

        PageRank semantics: a node (token) with many high-weight INCOMING edges
        is one that many tokens attend to — exactly the tokens worth keeping.

        Returns: (PyDiGraph, list of node indices)
        """
        G = rx.PyDiGraph()
        node_ids = G.add_nodes_from(list(range(n)))

        edges = []
        for src in range(n):
            row = attn_rows[src]                      # weights src placed on 0..src
            dst_len = min(len(row), src + 1)          # causal: dst ≤ src
            for dst in range(dst_len):
                w = float(row[dst].item())
                if w > 1e-4:                          # threshold (1e-4 not 1e-6)
                    edges.append((src, dst, w))       # src attends to dst

        if edges:
            G.add_edges_from(edges)

        return G, node_ids

    # ------------------------------------------------------------------
    def _flush_with_pagerank(self):
        """
        Move the oldest half of the recent zone to the keyword zone.

        Steps:
          1. Take oldest `flush_n` tokens.
          2. Build rustworkx PyDiGraph with attention weights.
          3. Run PageRank to get importance scores.
          4. Keep top KEYWORD_RATIO fraction; discard the rest.
        """
        flush_n = self.window // 2
        flush_K    = self.recent_K[:flush_n]
        flush_V    = self.recent_V[:flush_n]
        flush_rows = self.recent_attn_rows[:flush_n]

        # ── Build attention graph ──────────────────────────────────────
        G, node_ids = self._build_attention_graph(flush_rows, flush_n)

        # ── PageRank on the attention graph ───────────────────────────
        # weight_fn maps edge payload (float) → float weight for PageRank
        try:
            pr_map = rx.pagerank(G, alpha=0.85, weight_fn=lambda e: float(e))
        except Exception:
            # Fallback: if graph has no edges, uniform scores
            pr_map = {i: 1.0 / flush_n for i in range(flush_n)}

        self.pagerank_calls += 1
        pr_keys   = set(pr_map.keys())
        pr_scores = [pr_map[i] if i in pr_keys else 0.0 for i in range(flush_n)]
        self.last_pr_scores = pr_scores   # store for inspection

        # ── Select top-k% by PageRank ─────────────────────────────────
        keep_n   = max(1, int(flush_n * self.ratio))
        ranked   = sorted(range(flush_n), key=lambda i: pr_scores[i], reverse=True)
        keep_idx = set(ranked[:keep_n])

        for i in range(flush_n):
            if i in keep_idx:
                self.keyword_K.append(flush_K[i])
                self.keyword_V.append(flush_V[i])
            # else: discarded — true memory saving

        # ── Trim recent zone ──────────────────────────────────────────
        self.recent_K         = self.recent_K[flush_n:]
        self.recent_V         = self.recent_V[flush_n:]
        self.recent_attn_rows = self.recent_attn_rows[flush_n:]

    # ------------------------------------------------------------------
    def get_kv(self):
        """Return full usable K, V tensors: keywords + recent. [n_kept, d_head]"""
        all_K = self.keyword_K + self.recent_K
        all_V = self.keyword_V + self.recent_V
        if not all_K:
            return None, None
        K = torch.stack(all_K).to(self.device)
        V = torch.stack(all_V).to(self.device)
        return K, V

    # ------------------------------------------------------------------
    def memory_stats(self):
        full_n   = self.total_tokens
        stored_n = len(self.keyword_K) + len(self.recent_K)
        saved    = (1 - stored_n / max(full_n, 1)) * 100
        return {
            'total':          full_n,
            'stored':         stored_n,
            'keywords':       len(self.keyword_K),
            'recent':         len(self.recent_K),
            'saved_pct':      saved,
            'pagerank_calls': self.pagerank_calls,
        }


# ── Quick smoke test ──────────────────────────────────────────────────────────
def _smoke_test_keyword_cache():
    device = torch.device('cpu')
    d      = 64
    cache  = KeywordKVCache(window=16, ratio=0.3, d_head=d, device=device)

    torch.manual_seed(0)
    # Simulate 40 tokens with random K/V and random attention columns
    for t in range(40):
        k = torch.randn(d)
        v = torch.randn(d)
        # Simulate a causal attention column for this token
        col = torch.softmax(torch.randn(t + 1), dim=0)
        cache.update(k, v, attn_row=col)

    K, V = cache.get_kv()
    stats = cache.memory_stats()
    print(f"Smoke test: 40 tokens → stored={stats['stored']} "
          f"(keywords={stats['keywords']}, recent={stats['recent']}), "
          f"saved={stats['saved_pct']:.0f}%, "
          f"PageRank calls={stats['pagerank_calls']}")
    print(f"  K shape: {K.shape}, V shape: {V.shape}")
    print(f"  Last PageRank scores (first 8): "
          f"{[f'{s:.4f}' for s in (cache.last_pr_scores or [])[:8]]}")
    assert K.shape[0] == stats['stored'], "K shape mismatch"
    print("  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed")

_smoke_test_keyword_cache()
print("\nIdea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓")

Smoke test: 40 tokens → stored=22 (keywords=6, recent=16), saved=45%, PageRank calls=3
  K shape: torch.Size([22, 64]), V shape: torch.Size([22, 64])
  Last PageRank scores (first 8): ['0.7251', '0.0789', '0.0552', '0.0356', '0.0333', '0.0241', '0.0255', '0.0223']
  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed

Idea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓


## Idea 3 — V Deduplication (Retired)

**Why it fails for GPT-2:** `V = x @ W_V^T` where `x = tok_emb[id] + pos_emb[pos] + residual`.
The residual stream dominates by layer 3+ — two occurrences of the same token ID have
essentially unrelated V vectors in deeper layers. PPL went from 34.79 → 3258 in ablation.

**Where it works:** RoPE models (LLaMA, Mistral, Qwen) apply rotation only to Q and K.
V is computed from token embeddings alone, making it position-independent and exactly deduplicable.

This idea is kept as a documented stub for RoPE model implementations.


In [37]:
# ── Idea 3: Retired — stub only ──────────────────────────────────────────────
# Works on RoPE models only. See markdown above for explanation.

class VDeduplicator:
    """Stub — no-op for GPT-2. Exact implementation valid for LLaMA/Mistral."""
    def __init__(self): pass
    def get_or_store(self, token_id, v): return v

print("Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.")


Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.


## Idea 4 — Diagonal W_K / W_V: Exact V Recovery

**Key insight:** If we restrict `W_K` and `W_V` to diagonal matrices:
```
K_i = w_k[i] * x_i      (element-wise scaling of input dimension i)
V_i = w_v[i] * x_i
→ V_i = K_i * (w_v[i] / w_k[i])    for all i
→ V   = K * r            where  r = w_v / w_k   (d_head scalars, per-head constant)
```

**At inference:** Cache only K (n × d_head). Reconstruct V on the fly as `K * r`.
Store r once per head — negligible memory (64 floats for d_head=64).
**Result: exact 50% KV cache reduction, zero approximation error.**

**W_Q stays full-rank:** Q is computed fresh each forward step, never cached.
Restricting W_Q would hurt attention quality with no memory benefit.

**Training:** Must train from scratch. The diagonal constraint is enforced by
representing W_K and W_V as `nn.Parameter` vectors (not matrices).
Gradient flow is identical to standard attention — just fewer parameters.

**Expressivity tradeoff:**
- Standard head: `d_model × d_head` params each for K and V = 768×64 = 49,152 each
- Diagonal head: `d_head` params each for K and V = 64 each
- Q unchanged: `d_model × d_head` = 49,152 params
- Per-layer saving: ~98% of K+V params (but Q dominates — overall ~33% attention param reduction)

**What diagonal K/V heads learn:** Each head learns which input dimensions to
amplify or suppress when routing (K) and retrieving (V). Different heads specialise
on different feature dimensions. Less flexible than full-rank but interpretable.


In [38]:
# ── Idea 4: Diagonal W_K / W_V Attention ─────────────────────────────────────
#
# W_Q: full-rank nn.Linear (d_model → d_head)  — unchanged, Q not cached
# W_K: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_k)
# W_V: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_v)
#   but x is [B, T, d_model] and d_model != d_head, so we need a projection.
#   Solution: W_K_proj: Linear(d_model → d_head, bias=False) THEN diagonal scale.
#   This keeps the input projection (for dimensionality reduction) but the
#   "rotation" in head space is replaced by a diagonal scale.
#
# Memory at inference:
#   Standard: cache K [n, d_head] + V [n, d_head] = 2n*d_head values
#   Diagonal: cache K [n, d_head] + r [d_head]    = n*d_head + d_head values
#   Saving:   ~50% (r is negligible vs n*d_head for long contexts)
#
# V recovery: V = K * r  where r = w_v / w_k  (element-wise, broadcast over n)
#   This is EXACT — no approximation, no fine-tuning needed.

class DiagonalKVAttention(nn.Module):
    """
    Causal attention with diagonal W_K and W_V in head space.

    Architecture:
      Q = x @ W_Q^T                    (full-rank, d_model → d_head)
      K = (x @ W_K_proj^T) * w_k       (project then diagonal-scale)
      V = K * r    where r = w_v/w_k   (exact recovery, no storage needed)

    KV cache at inference:
      Store: K [n, d_head]  +  r [d_head]  (shared across all n tokens)
      Recover: V = K * r  (one elementwise multiply, O(n*d_head))
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank projection (not cached — no reason to restrict)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: linear projection into head space, then diagonal scale
        # W_K_proj reduces d_model → d_head (the expensive part, kept for expressivity)
        # w_k is the diagonal scale applied after projection (d_head params)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))   # diagonal K scale

        # V: diagonal scale only — w_v / w_k gives the recovery ratio r
        self.w_v      = nn.Parameter(torch.ones(d_out))   # diagonal V scale

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        # Initialise w_k and w_v with small random values (like default Linear init)
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)                          # [B, T, d_head]  full-rank
        k = self.W_k_proj(x) * self.w_k          # [B, T, d_head]  diagonal-scaled
        v = k * self.r                            # [B, T, d_head]  exact recovery

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        """Report memory usage vs standard attention."""
        return {'r': self.r.detach().cpu(),
                'w_k_range': (self.w_k.min().item(), self.w_k.max().item()),
                'w_v_range': (self.w_v.min().item(), self.w_v.max().item())}


class MHADiagonalKV(nn.Module):
    """Multi-head attention with diagonal K/V and full-rank Q."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass   # XSA not used on diagonal heads (trained from scratch)

print("Idea 4 — DiagonalKVAttention defined ✓")
print(f"  W_Q: full-rank  (d_model × d_head)")
print(f"  W_K: proj + diagonal scale  (d_model×d_head + d_head params)")
print(f"  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)")
print(f"  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error")


Idea 4 — DiagonalKVAttention defined ✓
  W_Q: full-rank  (d_model × d_head)
  W_K: proj + diagonal scale  (d_model×d_head + d_head params)
  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)
  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error


## Idea 4b — Givens Rotation + Diagonal KV

**Motivation:** Diagonal `W_K`/`W_V` recovers 50% KV memory exactly, but loses all rotational expressivity —
each head can only scale dimensions, not mix them.
A learnable rotation before the diagonal scale recovers some of that expressivity
without breaking the exact V-recovery property.

**The key constraint:** V recovery requires `V = K * r` for a fixed vector `r`.
This holds if and only if K and V share the *same* linear transformation before
their diagonal scales:

```
x_proj = W_K_proj(x)            # shared full-rank projection  (d_model → d_head)
K      = (x_proj @ R.T) * w_k   # shared rotation R, then diagonal scale w_k
V      = K * r                   # exact: r = w_v / w_k   ✓
```

The rotation `R` must be the **same** for K and V — otherwise V ≠ K * r.
If both use the same `R`, then `V = diag(w_v) @ R @ x_proj = diag(w_v/w_k) @ K`,
so exact recovery still holds.

**Why not a full orthogonal matrix?**
A full `d_head × d_head` rotation has `d_head*(d_head-1)/2 = 2016` parameters
(for d_head=64) — equivalent to an unconstrained `W_K`. That defeats the purpose.

**Givens rotations — the sweet spot:**
A Givens rotation `R(i, j, θ)` rotates the 2D subspace spanned by axes `i` and `j`
by angle `θ`, leaving all other axes unchanged. It is parameterised by a single
learned scalar `θ`. A product of `p` Givens rotations:

- Gives `p` free parameters (the angles `θ₁, …, θₚ`)
- Is always an exact orthogonal matrix — no normalisation overhead
- Composes cheaply: each step is one 2D rotation (O(n) not O(n²))
- Interpolates between diagonal (p=0) and full rotation (p = d_head*(d_head-1)/2)

**Architecture:**

```
x_proj  = W_K_proj(x)               # d_model → d_head  (full-rank, shared)
x_rot   = GivensRotation(x_proj)     # p learned Givens rotations applied in sequence
K       = x_rot * w_k               # diagonal scale
V       = K * r                      # exact recovery, r = w_v / w_k
```

**Parameter count (per head, d_head=64):**
| Config               | Extra params | Notes                              |
|---------------------|--------------|------------------------------------|
| p=0  (diagonal only) | 0            | baseline Idea 4                   |
| p=32 (half sweep)    | 32           | each dimension-pair rotated once   |
| p=64 (one sweep)     | 64           | recommended default                |
| p=128 (two sweeps)   | 128          | two passes, richer mixing          |
| p=2016 (full group)  | 2016         | equivalent to full W_K (factored)  |

**V recovery proof:**
```
K = diag(w_k) @ R @ x_proj      # R = product of Givens rotations
V = diag(w_v) @ R @ x_proj      # same R applied
  = diag(w_v) @ diag(1/w_k) @ diag(w_k) @ R @ x_proj
  = diag(w_v / w_k) @ K
  = K * r                        # exact ✓
```

**Initialisation:** Angles `θ` initialised to 0 → identity rotation at start of training.
The model begins as a pure diagonal model and learns to rotate as needed.

In [39]:
# ── Idea 4b: GivensRotation + DiagonalKV ─────────────────────────────────────
#
# A GivensRotation module applies p sequential 2D rotations in learned (i,j) planes.
# Each rotation is parameterised by one learned angle θ (initialised to 0 → identity).
#
# Architecture per head:
#   x_proj = W_K_proj(x)               [B, T, d_head]  full-rank (shared)
#   x_rot  = GivensRotation(x_proj)    [B, T, d_head]  p learned rotations
#   K      = x_rot * w_k               [B, T, d_head]  diagonal scale
#   V      = K * r                     [B, T, d_head]  exact recovery (r = w_v/w_k)
#
# V recovery proof:
#   K = diag(w_k) @ R @ x_proj
#   V = diag(w_v) @ R @ x_proj  (same R)
#     = diag(w_v/w_k) @ K = K * r   ✓
#
# n_givens_pairs controls expressivity:
#   0    → pure diagonal (Idea 4 baseline)
#   d//2 → one half-sweep (all pairs once, non-overlapping)
#   d    → one full sweep (recommended default, 64 params for d_head=64)
#   d*(d-1)//2 → full rotation group (equivalent to full-rank W_K, factored)

import itertools

class GivensRotation(nn.Module):
    """
    Learnable product of p Givens (plane) rotations applied to the last dimension.

    Each rotation acts in a 2D subspace (axes i, j) by angle θ:
        x[..., i] ← x[..., i]*cos(θ) - x[..., j]*sin(θ)
        x[..., j] ← x[..., i]*sin(θ) + x[..., j]*cos(θ)

    The p (i, j) plane pairs are fixed at construction (sequential sweep of consecutive
    pairs: (0,1), (2,3), ..., then (1,2), (3,4), ... for p > d//2).
    Only the angles θ are learned.

    Initialisation: θ = 0  →  identity at the start of training.
    The model starts as a pure diagonal model and grows rotational structure as needed.

    Args:
        d_head        : dimension being rotated
        n_pairs       : number of Givens rotations (p). 0 = identity (no-op).
    """
    def __init__(self, d_head: int, n_pairs: int):
        super().__init__()
        self.d_head  = d_head
        self.n_pairs = n_pairs

        if n_pairs == 0:
            # No-op: register no parameters, forward is an identity.
            self.register_buffer('_dummy', torch.zeros(1))
            return

        # ── Generate plane pairs ──────────────────────────────────────────
        # Sweep consecutive pairs in multiple passes until we have n_pairs.
        # Pass 0: (0,1), (2,3), (4,5), ...         (non-overlapping, parallelisable)
        # Pass 1: (1,2), (3,4), (5,6), ...
        # Pass 2: (0,1), (2,3), ...  (repeat)
        # This covers all adjacent pairs before reusing any, giving diverse mixing.
        planes = []
        pass_idx = 0
        while len(planes) < n_pairs:
            offset = pass_idx % 2          # alternates 0 and 1
            start  = offset
            for i in range(start, d_head - 1, 2):
                planes.append((i, i + 1))
                if len(planes) == n_pairs:
                    break
            pass_idx += 1

        self.planes = planes               # list of (i, j) tuples, length n_pairs

        # Learned angles — initialised to 0 (identity rotation)
        self.angles = nn.Parameter(torch.zeros(n_pairs))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [..., d_head]
        Returns rotated tensor of same shape.
        """
        if self.n_pairs == 0:
            return x

        # Clone to avoid in-place autograd issues across loop iterations
        x = x.clone()

        for k, (i, j) in enumerate(self.planes):
            theta = self.angles[k]
            cos_t = torch.cos(theta)
            sin_t = torch.sin(theta)
            xi = x[..., i].clone()
            xj = x[..., j].clone()
            x[..., i] = cos_t * xi - sin_t * xj
            x[..., j] = sin_t * xi + cos_t * xj

        return x

    def extra_repr(self) -> str:
        return f"d_head={self.d_head}, n_pairs={self.n_pairs}"


# ── GivensKVAttention — drop-in replacement for DiagonalKVAttention ───────────

class GivensKVAttention(nn.Module):
    """
    DiagonalKVAttention augmented with a shared learnable Givens rotation.

    Architecture:
      Q      = W_Q(x)                              full-rank (d_model → d_head)
      x_proj = W_K_proj(x)                         shared projection
      x_rot  = GivensRotation(x_proj)              p learned rotations
      K      = x_rot * w_k                         diagonal scale
      V      = K * r   where r = w_v / w_k         exact recovery ✓

    KV cache at inference:
      Store K [n, d_head]  +  r [d_head]   →  same 50% saving as Idea 4.
      Reconstruction: V = K * r  (unchanged from Idea 4).

    Args:
      d_in, d_out   : as per DiagonalKVAttention
      n_givens_pairs: number of Givens rotations (0 = pure diagonal baseline)
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank (not cached)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: project, rotate (Givens), diagonal scale
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.givens   = GivensRotation(d_out, n_givens_pairs)
        self.w_k      = nn.Parameter(torch.ones(d_out))

        # V: exact recovery from K
        self.w_v      = nn.Parameter(torch.ones(d_out))

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)
        # angles initialised to 0 inside GivensRotation → identity start

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape

        q      = self.W_q(x)                        # [B, T, d_head]
        x_proj = self.W_k_proj(x)                   # [B, T, d_head]  shared projection
        x_rot  = self.givens(x_proj)                # [B, T, d_head]  Givens rotation
        k      = x_rot * self.w_k                   # [B, T, d_head]  diagonal scale
        v      = k * self.r                         # [B, T, d_head]  exact V recovery

        s    = (q @ k.transpose(1, 2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T, :T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        return {
            'r':           self.r.detach().cpu(),
            'w_k_range':   (self.w_k.min().item(), self.w_k.max().item()),
            'w_v_range':   (self.w_v.min().item(), self.w_v.max().item()),
            'angles_std':  self.givens.angles.std().item() if self.givens.n_pairs > 0 else 0.0,
        }


class MHAGivensKV(nn.Module):
    """Multi-head attention using GivensKVAttention heads."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads,
                 qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.heads = nn.ModuleList([
            GivensKVAttention(d_model, d_head, ctx, drop, qkv_bias,
                              n_givens_pairs=n_givens_pairs)
            for _ in range(n_heads)
        ])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass  # XSA not used here


def make_givens_block_cls(cfg, n_givens_pairs=64):
    """Factory: returns a Block class with GivensKV attention."""
    d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']

    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = MHAGivensKV(d, dh, cfg['context_length'], cfg['drop'],
                                    cfg['n_head'], cfg['qkv_bias'],
                                    n_givens_pairs=n_givens_pairs)
            self.ffn  = FeedForward(cfg)

        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x)))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x

    return Block


print("Idea 4b — GivensKVAttention defined ✓")
print(f"  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale")
print(f"  V recovery:     V = K * r  (unchanged — exact, zero approximation error)")
print(f"  KV cache:       store K only + r per head  →  50% saving preserved")
print(f"  Init:           angles=0  →  identity at t=0  (pure diagonal start)")
print()
print("Plane generation for d_head=64, n_pairs=8:")
_demo = GivensRotation(64, 8)
print(f"  planes = {_demo.planes}")
print(f"  angles = {_demo.angles.data.tolist()}  (all zeros → identity)")


Idea 4b — GivensKVAttention defined ✓
  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale
  V recovery:     V = K * r  (unchanged — exact, zero approximation error)
  KV cache:       store K only + r per head  →  50% saving preserved
  Init:           angles=0  →  identity at t=0  (pure diagonal start)

Plane generation for d_head=64, n_pairs=8:
  planes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15)]
  angles = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  (all zeros → identity)


### Expressivity Sweep — Givens Pairs vs Perplexity

We train a small GPT-2-scale model for each value of `n_givens_pairs` and compare
the final perplexity after a fixed number of steps. This shows how much the rotation
budget matters and finds the sweet spot between diagonal simplicity and full-rank power.

**Sweep values:** `p ∈ {0, 8, 32, 64, 128}` pairs
- `p=0`: baseline Idea 4 (pure diagonal)
- `p=64`: recommended default (one full adjacent sweep for d_head=64)
- `p=128`: two sweeps — richer mixing at still modest parameter cost

In [40]:
# ── Definitions inlined for standalone execution ────────────────────────────
import math, urllib.request, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from transformers import GPT2Model
import rustworkx as rx           # ← Rust-backed graph library for PageRank

warnings.filterwarnings('ignore')
torch.manual_seed(42)

MAX_TOKENS       = 4096
KEYWORD_RATIO    = 0.3   # Idea 2: keep top 30% of tokens after threshold
KEYWORD_WINDOW   = 256   # Idea 2: tokens before this = full cache; after = keyword only
DEDUP_ENABLED    = True  # Idea 3: deduplicate V by token id
EIGENBASIS_RANK  = 32    # Idea 4: rank-r approx of shared eigenbasis (None = disabled)

# ── Shared layers (unchanged from original) ───────────────────────────────────

class LayerNorm(nn.Module):
    """Fixed: unbiased=False variance matches GPT-2 paper."""
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta  = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        return (x - mean) / (var + self.eps).sqrt() * self.gamma + self.beta

class GeLU(nn.Module):
    def forward(self, x): return F.gelu(x)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['emb_dim']
        self.layers = nn.Sequential(nn.Linear(d, 4*d), GeLU(), nn.Linear(4*d, d))
    def forward(self, x): return self.layers(x)


# ── Baseline: standard softmax attention (original notebook) ──────────────────

class CausalAttentionSoftmax(nn.Module):
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        return self.drop(torch.softmax(s, dim=-1)) @ v

class MHASoftmax(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionSoftmax(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)
    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))


# ── Idea 1: XSA — Exclusive Self-Attention ────────────────────────────────────
# Zhai, Apple 2026 (arXiv:2603.09078)
#
# IMPORTANT: The XSA projection is learned behaviour — it must be trained in
# from scratch.  Applying it post-hoc to GPT-2 pretrained weights removes signal
# the model already learnt to rely on, so PPL rises.
# We therefore use apply_xsa=False for perplexity measurement and apply_xsa=True
# only for the attention-sink analysis where we demonstrate the projection works.

class CausalAttentionXSA(nn.Module):
    """
    Standard causal attention + XSA projection step.
    After computing y = softmax(QK^T/√d)·V, we project out the
    component of y along the token's own value vector v_i:

        z_i = y_i - (y_i · v̂_i) · v̂_i

    This eliminates attention similarity bias / attention sinks.
    Use apply_xsa=False at inference on pretrained weights (PPL measurement).
    Use apply_xsa=True only when the model was trained with XSA from scratch.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.apply_xsa = False   # ← set True only when trained from scratch

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = self.drop(torch.softmax(s, dim=-1))
        y = attn @ v

        if self.apply_xsa:
            # Remove self-value projection: z_i = y_i - (y_i·v̂_i)·v̂_i
            v_norm = F.normalize(v, dim=-1)
            proj   = (y * v_norm).sum(dim=-1, keepdim=True)
            y      = y - proj * v_norm

        return y

class MHAXSA(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionXSA(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def set_apply_xsa(self, flag: bool):
        for h in self.heads:
            h.apply_xsa = flag

    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]")


# ── Idea 2: Keyword Threshold KV Cache — rustworkx PageRank ──────────────────
#
# KEY INSIGHT: Build a directed attention graph for the flush batch using
# rustworkx.PyDiGraph.  Edge i→j carries weight = attention_weight[i,j].
# Run rustworkx.pagerank() — tokens that many others attend to get high score.
# Keep top KEYWORD_RATIO fraction by PageRank score; discard the rest.
#
# rustworkx is a Rust-backed Python graph library (used in Qiskit).
# It is significantly faster than networkx for this workload.

import rustworkx as rx

class KeywordKVCache:
    """
    Manages a two-zone KV cache with rustworkx PageRank importance scoring.

      Zone A (recent): last `window` tokens → full exact K, V stored.
      Zone B (old):    tokens beyond window → only top-k% 'keywords' kept.

    Importance scoring (flush time):
      1. Build PyDiGraph: node per token, edge i→j weighted by attn_weight[i,j].
      2. Run rustworkx.pagerank(G, alpha=0.85, weight_fn=lambda e: e).
      3. Rank nodes by PageRank score; keep top KEYWORD_RATIO fraction.

    Usage:
      cache = KeywordKVCache(window=256, ratio=0.3, d_head=64, device=device)
      cache.update(k_vec, v_vec, attn_row)   # one token at a time
      K, V  = cache.get_kv()                 # retrieve full usable KV tensors
      stats = cache.memory_stats()
    """

    def __init__(self, window: int, ratio: float, d_head: int, device):
        self.window  = window
        self.ratio   = ratio
        self.d_head  = d_head
        self.device  = device

        # Zone A — recent tokens (full K, V stored)
        self.recent_K    = []   # list of [d_head] CPU tensors
        self.recent_V    = []
        # attn_rows[t] = the attention ROW for token t:
        #   attn_rows[t][j] = weight that token t placed on token j (j ≤ t).
        # This is the row of the attention matrix, NOT the column.
        self.recent_attn_rows = []   # list of 1-D tensors, variable length

        # Zone B — evicted keyword tokens
        self.keyword_K = []
        self.keyword_V = []

        self.total_tokens       = 0
        self.pagerank_calls     = 0
        self.last_pr_scores     = None   # PageRank scores from last flush (for inspection)

    # ------------------------------------------------------------------
    def update(self, k: torch.Tensor, v: torch.Tensor,
               attn_row: torch.Tensor = None):
        """
        Add one token's K and V to the cache.

        attn_row: 1-D tensor of length (t+1) giving the attention weights
                  that THIS token placed on tokens 0..t-1 (i.e. row t of the
                  causal attention matrix: attn[t, 0..t]).
                  If None, a uniform weight of 1/n is assumed (degrades to
                  uniform PageRank, still correct).
        """
        self.recent_K.append(k.detach().cpu())
        self.recent_V.append(v.detach().cpu())

        n = len(self.recent_K)
        if attn_row is not None:
            row = attn_row.detach().cpu()
        else:
            row = torch.full((n,), 1.0 / n)

        self.recent_attn_rows.append(row)
        self.total_tokens += 1

        if len(self.recent_K) > self.window:
            self._flush_with_pagerank()

    # ------------------------------------------------------------------
    def _build_attention_graph(self, attn_rows, n):
        """
        Build a rustworkx PyDiGraph from the causal attention matrix.

        attn_rows[src] is the ATTENTION ROW for token `src`:
          attn_rows[src][dst] = weight that token `src` placed on token `dst`.
        Causal constraint: dst <= src (future tokens are not attended to).

        Edge direction: src → dst  ("token src attends to token dst")
        Weight = attention_weight[src, dst].

        PageRank semantics: a node (token) with many high-weight INCOMING edges
        is one that many tokens attend to — exactly the tokens worth keeping.

        Returns: (PyDiGraph, list of node indices)
        """
        G = rx.PyDiGraph()
        node_ids = G.add_nodes_from(list(range(n)))

        edges = []
        for src in range(n):
            row = attn_rows[src]                      # weights src placed on 0..src
            dst_len = min(len(row), src + 1)          # causal: dst ≤ src
            for dst in range(dst_len):
                w = float(row[dst].item())
                if w > 1e-4:                          # threshold (1e-4 not 1e-6)
                    edges.append((src, dst, w))       # src attends to dst

        if edges:
            G.add_edges_from(edges)

        return G, node_ids

    # ------------------------------------------------------------------
    def _flush_with_pagerank(self):
        """
        Move the oldest half of the recent zone to the keyword zone.

        Steps:
          1. Take oldest `flush_n` tokens.
          2. Build rustworkx PyDiGraph with attention weights.
          3. Run PageRank to get importance scores.
          4. Keep top KEYWORD_RATIO fraction; discard the rest.
        """
        flush_n = self.window // 2
        flush_K    = self.recent_K[:flush_n]
        flush_V    = self.recent_V[:flush_n]
        flush_rows = self.recent_attn_rows[:flush_n]

        # ── Build attention graph ──────────────────────────────────────
        G, node_ids = self._build_attention_graph(flush_rows, flush_n)

        # ── PageRank on the attention graph ───────────────────────────
        # weight_fn maps edge payload (float) → float weight for PageRank
        try:
            pr_map = rx.pagerank(G, alpha=0.85, weight_fn=lambda e: float(e))
        except Exception:
            # Fallback: if graph has no edges, uniform scores
            pr_map = {i: 1.0 / flush_n for i in range(flush_n)}

        self.pagerank_calls += 1
        pr_keys   = set(pr_map.keys())
        pr_scores = [pr_map[i] if i in pr_keys else 0.0 for i in range(flush_n)]
        self.last_pr_scores = pr_scores   # store for inspection

        # ── Select top-k% by PageRank ─────────────────────────────────
        keep_n   = max(1, int(flush_n * self.ratio))
        ranked   = sorted(range(flush_n), key=lambda i: pr_scores[i], reverse=True)
        keep_idx = set(ranked[:keep_n])

        for i in range(flush_n):
            if i in keep_idx:
                self.keyword_K.append(flush_K[i])
                self.keyword_V.append(flush_V[i])
            # else: discarded — true memory saving

        # ── Trim recent zone ──────────────────────────────────────────
        self.recent_K         = self.recent_K[flush_n:]
        self.recent_V         = self.recent_V[flush_n:]
        self.recent_attn_rows = self.recent_attn_rows[flush_n:]

    # ------------------------------------------------------------------
    def get_kv(self):
        """Return full usable K, V tensors: keywords + recent. [n_kept, d_head]"""
        all_K = self.keyword_K + self.recent_K
        all_V = self.keyword_V + self.recent_V
        if not all_K:
            return None, None
        K = torch.stack(all_K).to(self.device)
        V = torch.stack(all_V).to(self.device)
        return K, V

    # ------------------------------------------------------------------
    def memory_stats(self):
        full_n   = self.total_tokens
        stored_n = len(self.keyword_K) + len(self.recent_K)
        saved    = (1 - stored_n / max(full_n, 1)) * 100
        return {
            'total':          full_n,
            'stored':         stored_n,
            'keywords':       len(self.keyword_K),
            'recent':         len(self.recent_K),
            'saved_pct':      saved,
            'pagerank_calls': self.pagerank_calls,
        }


# ── Quick smoke test ──────────────────────────────────────────────────────────
def _smoke_test_keyword_cache():
    device = torch.device('cpu')
    d      = 64
    cache  = KeywordKVCache(window=16, ratio=0.3, d_head=d, device=device)

    torch.manual_seed(0)
    # Simulate 40 tokens with random K/V and random attention columns
    for t in range(40):
        k = torch.randn(d)
        v = torch.randn(d)
        # Simulate a causal attention column for this token
        col = torch.softmax(torch.randn(t + 1), dim=0)
        cache.update(k, v, attn_row=col)

    K, V = cache.get_kv()
    stats = cache.memory_stats()
    print(f"Smoke test: 40 tokens → stored={stats['stored']} "
          f"(keywords={stats['keywords']}, recent={stats['recent']}), "
          f"saved={stats['saved_pct']:.0f}%, "
          f"PageRank calls={stats['pagerank_calls']}")
    print(f"  K shape: {K.shape}, V shape: {V.shape}")
    print(f"  Last PageRank scores (first 8): "
          f"{[f'{s:.4f}' for s in (cache.last_pr_scores or [])[:8]]}")
    assert K.shape[0] == stats['stored'], "K shape mismatch"
    print("  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed")

_smoke_test_keyword_cache()
print("\nIdea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓")

# ── Idea 3: Retired — stub only ──────────────────────────────────────────────
# Works on RoPE models only. See markdown above for explanation.

class VDeduplicator:
    """Stub — no-op for GPT-2. Exact implementation valid for LLaMA/Mistral."""
    def __init__(self): pass
    def get_or_store(self, token_id, v): return v

print("Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.")


# ── Idea 4: Diagonal W_K / W_V Attention ─────────────────────────────────────
#
# W_Q: full-rank nn.Linear (d_model → d_head)  — unchanged, Q not cached
# W_K: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_k)
# W_V: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_v)
#   but x is [B, T, d_model] and d_model != d_head, so we need a projection.
#   Solution: W_K_proj: Linear(d_model → d_head, bias=False) THEN diagonal scale.
#   This keeps the input projection (for dimensionality reduction) but the
#   "rotation" in head space is replaced by a diagonal scale.
#
# Memory at inference:
#   Standard: cache K [n, d_head] + V [n, d_head] = 2n*d_head values
#   Diagonal: cache K [n, d_head] + r [d_head]    = n*d_head + d_head values
#   Saving:   ~50% (r is negligible vs n*d_head for long contexts)
#
# V recovery: V = K * r  where r = w_v / w_k  (element-wise, broadcast over n)
#   This is EXACT — no approximation, no fine-tuning needed.

class DiagonalKVAttention(nn.Module):
    """
    Causal attention with diagonal W_K and W_V in head space.

    Architecture:
      Q = x @ W_Q^T                    (full-rank, d_model → d_head)
      K = (x @ W_K_proj^T) * w_k       (project then diagonal-scale)
      V = K * r    where r = w_v/w_k   (exact recovery, no storage needed)

    KV cache at inference:
      Store: K [n, d_head]  +  r [d_head]  (shared across all n tokens)
      Recover: V = K * r  (one elementwise multiply, O(n*d_head))
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank projection (not cached — no reason to restrict)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: linear projection into head space, then diagonal scale
        # W_K_proj reduces d_model → d_head (the expensive part, kept for expressivity)
        # w_k is the diagonal scale applied after projection (d_head params)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))   # diagonal K scale

        # V: diagonal scale only — w_v / w_k gives the recovery ratio r
        self.w_v      = nn.Parameter(torch.ones(d_out))   # diagonal V scale

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        # Initialise w_k and w_v with small random values (like default Linear init)
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)                          # [B, T, d_head]  full-rank
        k = self.W_k_proj(x) * self.w_k          # [B, T, d_head]  diagonal-scaled
        v = k * self.r                            # [B, T, d_head]  exact recovery

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        """Report memory usage vs standard attention."""
        return {'r': self.r.detach().cpu(),
                'w_k_range': (self.w_k.min().item(), self.w_k.max().item()),
                'w_v_range': (self.w_v.min().item(), self.w_v.max().item())}


class MHADiagonalKV(nn.Module):
    """Multi-head attention with diagonal K/V and full-rank Q."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass   # XSA not used on diagonal heads (trained from scratch)

print("Idea 4 — DiagonalKVAttention defined ✓")
print(f"  W_Q: full-rank  (d_model × d_head)")
print(f"  W_K: proj + diagonal scale  (d_model×d_head + d_head params)")
print(f"  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)")
print(f"  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error")


# ── Idea 4b: GivensRotation + DiagonalKV ─────────────────────────────────────
#
# A GivensRotation module applies p sequential 2D rotations in learned (i,j) planes.
# Each rotation is parameterised by one learned angle θ (initialised to 0 → identity).
#
# Architecture per head:
#   x_proj = W_K_proj(x)               [B, T, d_head]  full-rank (shared)
#   x_rot  = GivensRotation(x_proj)    [B, T, d_head]  p learned rotations
#   K      = x_rot * w_k               [B, T, d_head]  diagonal scale
#   V      = K * r                     [B, T, d_head]  exact recovery (r = w_v/w_k)
#
# V recovery proof:
#   K = diag(w_k) @ R @ x_proj
#   V = diag(w_v) @ R @ x_proj  (same R)
#     = diag(w_v/w_k) @ K = K * r   ✓
#
# n_givens_pairs controls expressivity:
#   0    → pure diagonal (Idea 4 baseline)
#   d//2 → one half-sweep (all pairs once, non-overlapping)
#   d    → one full sweep (recommended default, 64 params for d_head=64)
#   d*(d-1)//2 → full rotation group (equivalent to full-rank W_K, factored)

import itertools

class GivensRotation(nn.Module):
    """
    Learnable product of p Givens (plane) rotations applied to the last dimension.

    Each rotation acts in a 2D subspace (axes i, j) by angle θ:
        x[..., i] ← x[..., i]*cos(θ) - x[..., j]*sin(θ)
        x[..., j] ← x[..., i]*sin(θ) + x[..., j]*cos(θ)

    The p (i, j) plane pairs are fixed at construction (sequential sweep of consecutive
    pairs: (0,1), (2,3), ..., then (1,2), (3,4), ... for p > d//2).
    Only the angles θ are learned.

    Initialisation: θ = 0  →  identity at the start of training.
    The model starts as a pure diagonal model and grows rotational structure as needed.

    Args:
        d_head        : dimension being rotated
        n_pairs       : number of Givens rotations (p). 0 = identity (no-op).
    """
    def __init__(self, d_head: int, n_pairs: int):
        super().__init__()
        self.d_head  = d_head
        self.n_pairs = n_pairs

        if n_pairs == 0:
            # No-op: register no parameters, forward is an identity.
            self.register_buffer('_dummy', torch.zeros(1))
            return

        # ── Generate plane pairs ──────────────────────────────────────────
        # Sweep consecutive pairs in multiple passes until we have n_pairs.
        # Pass 0: (0,1), (2,3), (4,5), ...         (non-overlapping, parallelisable)
        # Pass 1: (1,2), (3,4), (5,6), ...
        # Pass 2: (0,1), (2,3), ...  (repeat)
        # This covers all adjacent pairs before reusing any, giving diverse mixing.
        planes = []
        pass_idx = 0
        while len(planes) < n_pairs:
            offset = pass_idx % 2          # alternates 0 and 1
            start  = offset
            for i in range(start, d_head - 1, 2):
                planes.append((i, i + 1))
                if len(planes) == n_pairs:
                    break
            pass_idx += 1

        self.planes = planes               # list of (i, j) tuples, length n_pairs

        # Learned angles — initialised to 0 (identity rotation)
        self.angles = nn.Parameter(torch.zeros(n_pairs))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [..., d_head]
        Returns rotated tensor of same shape.
        """
        if self.n_pairs == 0:
            return x

        # Clone to avoid in-place autograd issues across loop iterations
        x = x.clone()

        for k, (i, j) in enumerate(self.planes):
            theta = self.angles[k]
            cos_t = torch.cos(theta)
            sin_t = torch.sin(theta)
            xi = x[..., i].clone()
            xj = x[..., j].clone()
            x[..., i] = cos_t * xi - sin_t * xj
            x[..., j] = sin_t * xi + cos_t * xj

        return x

    def extra_repr(self) -> str:
        return f"d_head={self.d_head}, n_pairs={self.n_pairs}"


# ── GivensKVAttention — drop-in replacement for DiagonalKVAttention ───────────

class GivensKVAttention(nn.Module):
    """
    DiagonalKVAttention augmented with a shared learnable Givens rotation.

    Architecture:
      Q      = W_Q(x)                              full-rank (d_model → d_head)
      x_proj = W_K_proj(x)                         shared projection
      x_rot  = GivensRotation(x_proj)              p learned rotations
      K      = x_rot * w_k                         diagonal scale
      V      = K * r   where r = w_v / w_k         exact recovery ✓

    KV cache at inference:
      Store K [n, d_head]  +  r [d_head]   →  same 50% saving as Idea 4.
      Reconstruction: V = K * r  (unchanged from Idea 4).

    Args:
      d_in, d_out   : as per DiagonalKVAttention
      n_givens_pairs: number of Givens rotations (0 = pure diagonal baseline)
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank (not cached)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: project, rotate (Givens), diagonal scale
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.givens   = GivensRotation(d_out, n_givens_pairs)
        self.w_k      = nn.Parameter(torch.ones(d_out))

        # V: exact recovery from K
        self.w_v      = nn.Parameter(torch.ones(d_out))

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)
        # angles initialised to 0 inside GivensRotation → identity start

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape

        q      = self.W_q(x)                        # [B, T, d_head]
        x_proj = self.W_k_proj(x)                   # [B, T, d_head]  shared projection
        x_rot  = self.givens(x_proj)                # [B, T, d_head]  Givens rotation
        k      = x_rot * self.w_k                   # [B, T, d_head]  diagonal scale
        v      = k * self.r                         # [B, T, d_head]  exact V recovery

        s    = (q @ k.transpose(1, 2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T, :T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        return {
            'r':           self.r.detach().cpu(),
            'w_k_range':   (self.w_k.min().item(), self.w_k.max().item()),
            'w_v_range':   (self.w_v.min().item(), self.w_v.max().item()),
            'angles_std':  self.givens.angles.std().item() if self.givens.n_pairs > 0 else 0.0,
        }


class MHAGivensKV(nn.Module):
    """Multi-head attention using GivensKVAttention heads."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads,
                 qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.heads = nn.ModuleList([
            GivensKVAttention(d_model, d_head, ctx, drop, qkv_bias,
                              n_givens_pairs=n_givens_pairs)
            for _ in range(n_heads)
        ])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass  # XSA not used here


def make_givens_block_cls(cfg, n_givens_pairs=64):
    """Factory: returns a Block class with GivensKV attention."""
    d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']

    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = MHAGivensKV(d, dh, cfg['context_length'], cfg['drop'],
                                    cfg['n_head'], cfg['qkv_bias'],
                                    n_givens_pairs=n_givens_pairs)
            self.ffn  = FeedForward(cfg)

        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x)))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x

    return Block


print("Idea 4b — GivensKVAttention defined ✓")
print(f"  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale")
print(f"  V recovery:     V = K * r  (unchanged — exact, zero approximation error)")
print(f"  KV cache:       store K only + r per head  →  50% saving preserved")
print(f"  Init:           angles=0  →  identity at t=0  (pure diagonal start)")
print()
print("Plane generation for d_head=64, n_pairs=8:")
_demo = GivensRotation(64, 8)
print(f"  planes = {_demo.planes}")
print(f"  angles = {_demo.angles.data.tolist()}  (all zeros → identity)")


# ── Combined Attention: Diagonal KV (Idea 4) + XSA (Idea 1) ──────────────────

class DiagonalKVXSAAttention(nn.Module):
    """
    DiagonalKVAttention + XSA projection (Idea 1).
    Intended for training from scratch — apply_xsa=True always here.

    XSA removes self-value component from output:
        z_i = y_i - (y_i · v̂_i) · v̂_i
    With diagonal V = K * r, the self-value vector is well-defined
    and the projection is as meaningful as in full-rank attention.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out    = d_out
        self.W_q      = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))
        self.w_v      = nn.Parameter(torch.ones(d_out))
        self.drop     = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x, token_ids=None, v_dedup=None):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k_proj(x) * self.w_k
        v = k * self.r

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        y    = attn @ v

        # XSA: remove self-value component
        v_norm = F.normalize(v, dim=-1)
        proj   = (y * v_norm).sum(dim=-1, keepdim=True)
        y      = y - proj * v_norm
        return y


class MHACombined(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVXSAAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, token_ids=None, v_dedup=None):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Combined attention (Diagonal KV + XSA) defined ✓")


# ── GPT shell — same structure as original, works with any MHA class ──────────

def _block_cls(cfg, mha_cls, mha_kwargs=None):
    kwargs = mha_kwargs or {}
    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = mha_cls(d, dh, cfg['context_length'],
                                cfg['drop'], cfg['n_head'], cfg['qkv_bias'], **kwargs)
            self.ffn  = FeedForward(cfg)
        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x), **kw))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x
    return Block


class GPT(nn.Module):
    def __init__(self, cfg, block_cls):
        super().__init__()
        self.cfg         = cfg
        d                = cfg['emb_dim']
        self.tok_emb     = nn.Embedding(cfg['vocab_size'], d)
        self.pos_emb     = nn.Embedding(cfg['context_length'], d)
        self.drop        = nn.Dropout(cfg['drop'])
        self.trfs_blocks = nn.ModuleList([block_cls() for _ in range(cfg['n_layers'])])
        self.final_norm  = LayerNorm(d)
        self.lm_head     = nn.Linear(d, cfg['vocab_size'], bias=False)

    def forward(self, idx, v_dedup=None):
        # FIX: always pass idx (token IDs) to blocks so Idea 3 dedup can fire.
        # Previously token_ids was gated and never reached the attention heads.
        if idx.dim() == 1: idx = idx.unsqueeze(0)
        B, T = idx.shape
        T    = min(T, self.cfg['context_length'])
        idx  = idx[:, -T:]
        x    = self.drop(self.tok_emb(idx)
                       + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.trfs_blocks:
            # Pass token_ids=idx unconditionally; MHA classes that don't accept it
            # will ignore it (baseline/XSA heads don't have the parameter so we
            # guard by checking if the att module supports it).
            try:
                x = blk(x, token_ids=idx, v_dedup=v_dedup)
            except TypeError:
                x = blk(x)
        return self.lm_head(self.final_norm(x))

print("GPT shell defined ✓")


# ── Weight loaders ────────────────────────────────────────────────────────────
#
# load_gpt2_weights: loads into softmax baseline only (full-rank W_Q/K/V)
# init_diagonal_weights: random init for diagonal KV model (no pretrained weights)

def load_gpt2_weights(our_model, hf_model):
    """Load HuggingFace GPT-2 weights into the full-rank softmax baseline."""
    hf = hf_model.state_dict()
    sd = our_model.state_dict()

    sd['tok_emb.weight'].copy_(hf['wte.weight'])
    sd['pos_emb.weight'].copy_(hf['wpe.weight'])
    sd['final_norm.gamma'].copy_(hf['ln_f.weight'])
    sd['final_norm.beta' ].copy_(hf['ln_f.bias'  ])
    sd['lm_head.weight'  ].copy_(hf['wte.weight'])

    d, nh = our_model.cfg['emb_dim'], our_model.cfg['n_head']
    dh    = d // nh

    for i in range(our_model.cfg['n_layers']):
        p = f'h.{i}';  b = f'trfs_blocks.{i}'
        sd[f'{b}.ln1.gamma'].copy_(hf[f'{p}.ln_1.weight'])
        sd[f'{b}.ln1.beta' ].copy_(hf[f'{p}.ln_1.bias'  ])
        sd[f'{b}.ln2.gamma'].copy_(hf[f'{p}.ln_2.weight'])
        sd[f'{b}.ln2.beta' ].copy_(hf[f'{p}.ln_2.bias'  ])

        cw = hf[f'{p}.attn.c_attn.weight'].T
        cb = hf[f'{p}.attn.c_attn.bias'  ]
        Wq, Wk, Wv = cw[:d], cw[d:2*d], cw[2*d:]
        bq, bk, bv = cb[:d], cb[d:2*d], cb[2*d:]

        for h in range(nh):
            s, e = h*dh, (h+1)*dh
            hp   = f'{b}.att.heads.{h}'
            sd[f'{hp}.W_q.weight'].copy_(Wq[s:e])
            sd[f'{hp}.W_k.weight'].copy_(Wk[s:e])
            sd[f'{hp}.W_v.weight'].copy_(Wv[s:e])
            if our_model.cfg['qkv_bias']:
                sd[f'{hp}.W_q.bias'].copy_(bq[s:e])
                sd[f'{hp}.W_k.bias'].copy_(bk[s:e])
                sd[f'{hp}.W_v.bias'].copy_(bv[s:e])

        sd[f'{b}.att.out_proj.weight'].copy_(hf[f'{p}.attn.c_proj.weight'].T)
        sd[f'{b}.att.out_proj.bias'  ].copy_(hf[f'{p}.attn.c_proj.bias'  ])
        sd[f'{b}.ffn.layers.0.weight'].copy_(hf[f'{p}.mlp.c_fc.weight'   ].T)
        sd[f'{b}.ffn.layers.0.bias'  ].copy_(hf[f'{p}.mlp.c_fc.bias'     ])
        sd[f'{b}.ffn.layers.2.weight'].copy_(hf[f'{p}.mlp.c_proj.weight'  ].T)
        sd[f'{b}.ffn.layers.2.bias'  ].copy_(hf[f'{p}.mlp.c_proj.bias'   ])

    our_model.load_state_dict(sd)
    print("  ✓ Weights loaded")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("Weight loaders defined ✓")


# ── Perplexity evaluation ─────────────────────────────────────────────────────

@torch.no_grad()
def perplexity(model, token_ids, ctx, device, stride=128, use_dedup=False):
    """
    use_dedup: if True, enables the per-head VDeduplicator on combined_model heads.
               Dedup stores are reset at the start of each sliding window so that
               positional V vectors from one window don't bleed into the next.
               (Old behaviour: a single shared VDeduplicator was passed in from outside,
               causing cross-layer contamination and PPL=3258.  Fixed: each head owns
               its own store and it is reset per window.)
    """
    model.eval()
    token_ids = token_ids.to(device)
    T, nlls   = token_ids.size(1), []

    # Enable/disable per-head dedup
    if use_dedup:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'enable_dedup'):
                blk.att.enable_dedup()
    else:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'disable_dedup'):
                blk.att.disable_dedup()

    for begin in range(0, T - 1, stride):
        end  = min(begin + ctx, T)
        inp  = token_ids[:, begin:end-1]
        tgt  = token_ids[:, begin+1:end]
        if inp.size(1) == 0: break

        # Reset per-head V stores each window (avoid positional bleed across windows)
        if use_dedup:
            for blk in model.trfs_blocks:
                if hasattr(blk.att, 'reset_dedup'):
                    blk.att.reset_dedup()

        logits = model(inp, v_dedup=None)   # v_dedup arg ignored; heads are self-contained
        t      = logits.size(1)
        nlls.append(F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt[:, :t].reshape(-1)).item())

    # Leave dedup in whatever state caller wants after measurement
    return math.exp(sum(nlls) / len(nlls)), None


def get_text():
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    try:
        print("Fetching the-verdict.txt ...")
        with urllib.request.urlopen(url, timeout=15) as r:
            text = r.read().decode('utf-8')
        print(f"  Got {len(text):,} chars")
        return text
    except Exception as e:
        print(f"  Download failed ({e}), using fallback.")
        return "Every effort moves you forward. " * 500

print("Perplexity fn defined ✓")
print("  Fix: per-head dedup reset each window (no positional bleed)")


# ── Cell body ────────────────────────────────────────────────────────────────
# ── Sweep cell proper ────────────────────────────────────────────────────────
# ── Idea 4b: Expressivity Sweep — Givens pairs vs PPL ────────────────────────
#
# For each n_givens_pairs value, train a fresh GPT from scratch for TRAIN_STEPS
# steps and record final training loss, PPL, and angle statistics.
#
# Runtime note: each model trains for TRAIN_STEPS steps independently.
# On CPU this takes ~1-2 min per model for TRAIN_STEPS=300.
# Set TRAIN_STEPS smaller (e.g. 100) for a quick sanity check.

TRAIN_STEPS_GIVENS = 300   # ← increase to 500+ for more reliable comparison
SEQ_LEN_G          = 128
LR_G               = 3e-4

# p values to sweep.  p=0 is the pure-diagonal baseline (Idea 4).
N_GIVENS_SWEEP = [0, 8, 32, 64, 128]

def train_givens_model(cfg, token_ids, device, n_givens_pairs, steps, seq_len, lr):
    """Train a GivensKV GPT from scratch; return (model, final_ppl, losses)."""
    block_cls = make_givens_block_cls(cfg, n_givens_pairs=n_givens_pairs)
    model = GPT(cfg, block_cls).to(device)

    n_total = sum(p.numel() for p in model.parameters())
    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
                     optimizer, T_max=steps, eta_min=lr * 0.1)

    T_total = token_ids.size(1)
    model.train()
    losses = []

    for step in range(steps):
        start  = torch.randint(0, max(1, T_total - seq_len - 1), (1,)).item()
        inp    = token_ids[:, start:start + seq_len].to(device)
        tgt    = token_ids[:, start + 1:start + seq_len + 1].to(device)
        logits = model(inp)
        loss   = F.cross_entropy(logits.reshape(-1, cfg['vocab_size']),
                                  tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    model.eval()
    ppl, _ = perplexity(model, token_ids, cfg['context_length'], device, stride=64)
    return model, ppl, losses, n_total


def angle_stats(model):
    """Collect per-layer, per-head angle std across the whole model."""
    stds = []
    for blk in model.trfs_blocks:
        for h in blk.att.heads:
            if h.givens.n_pairs > 0:
                stds.append(h.givens.angles.std().item())
    return stds


print("Running Givens expressivity sweep ...")
print(f"  Steps per model : {TRAIN_STEPS_GIVENS}")
print(f"  p values        : {N_GIVENS_SWEEP}")
print()

# Self-contained setup — works whether or not run() has been called first.
import urllib.request, tiktoken as _tiktoken

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Always use a small config for the sweep so it runs quickly
sweep_standalone_cfg = {
    'vocab_size': 50257, 'n_head': 12, 'drop': 0.0,
    'n_layers': 2, 'context_length': 256,
    'qkv_bias': True, 'emb_dim': 768,
}

# Fetch the corpus (re-download if needed — cached by urllib)
def _fetch_text():
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            return r.read().decode('utf-8')
    except Exception:
        return "Every effort moves you forward. " * 500

_raw      = _fetch_text()
_enc      = _tiktoken.get_encoding("gpt2")
_all_ids  = _enc.encode(_raw, allowed_special={'<|endoftext|>'})
token_ids = torch.tensor(_all_ids, dtype=torch.long).unsqueeze(0)
print(f"Corpus: {token_ids.size(1):,} tokens")

# Override to a tiny model for the sweep so it runs fast everywhere
# sweep_cfg defined above as sweep_standalone_cfg
# config already set in sweep_standalone_cfg above

sweep_results = {}   # p → dict

for p in N_GIVENS_SWEEP:
    label = f"p={p:3d}"
    print(f"  [{label}]  Training {TRAIN_STEPS_GIVENS} steps ...", end=' ', flush=True)
    model_p, ppl_p, losses_p, n_params = train_givens_model(
        sweep_standalone_cfg, token_ids, device,
        n_givens_pairs=p,
        steps=TRAIN_STEPS_GIVENS,
        seq_len=SEQ_LEN_G,
        lr=LR_G,
    )
    avg_loss = sum(losses_p[-50:]) / min(50, len(losses_p))
    a_stds   = angle_stats(model_p)
    mean_astd = sum(a_stds) / len(a_stds) if a_stds else 0.0
    sweep_results[p] = {
        'ppl':        ppl_p,
        'avg_loss':   avg_loss,
        'n_params':   n_params,
        'angle_std':  mean_astd,
        'model':      model_p,
        'losses':     losses_p,
    }
    print(f"PPL={ppl_p:.2f}  loss={avg_loss:.4f}  angle_std={mean_astd:.4f}")

print()
print("=" * 68)
print(f"  {'p':>5}  {'PPL':>8}  {'loss':>8}  {'Δparams':>10}  {'angle_std':>10}")
print("=" * 68)
base_params = sweep_results[0]['n_params']
for p, r in sweep_results.items():
    delta = r['n_params'] - base_params
    print(f"  {p:>5}  {r['ppl']:>8.2f}  {r['avg_loss']:>8.4f}  "
          f"  +{delta:>7,}  {r['angle_std']:>10.4f}")
print("=" * 68)
print()
print("angle_std: how far learned angles deviate from 0 (identity).")
print("  Near 0  → rotation barely needed; diagonal already sufficient.")
print("  Growing → model actively uses the rotational degrees of freedom.")
print()
print("V-recovery sanity check (p=64 model, layer 0, head 0):")
with torch.no_grad():
    model64 = sweep_results[max(p for p in N_GIVENS_SWEEP if p > 0)]['model']
    h0      = model64.trfs_blocks[0].att.heads[0]
    dummy   = token_ids[:, :32].to(device)
    x_emb   = model64.drop(
        model64.tok_emb(dummy)
        + model64.pos_emb(torch.arange(32, device=device))
    )
    x_proj = h0.W_k_proj(x_emb)
    x_rot  = h0.givens(x_proj)
    k      = x_rot * h0.w_k
    v_rec  = k * h0.r                  # recovered V
    v_true = k * h0.r                  # same formula → must match
    err    = (v_rec - v_true).abs().max().item()
print(f"  Max |V_recovered - V_true| = {err:.2e}  (should be 0.00e+00)")
print()
print("KV cache behaviour (unchanged from Idea 4):")
print("  At inference: store K [n, d_head] + r [d_head] per head.")
print("  Givens rotation is absorbed into K during the forward pass —")
print("  it adds ZERO inference overhead to the KV cache size.")
print("  Memory saving: still exactly 50%, regardless of p.")


Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]
Smoke test: 40 tokens → stored=22 (keywords=6, recent=16), saved=45%, PageRank calls=3
  K shape: torch.Size([22, 64]), V shape: torch.Size([22, 64])
  Last PageRank scores (first 8): ['0.7251', '0.0789', '0.0552', '0.0356', '0.0333', '0.0241', '0.0255', '0.0223']
  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed

Idea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓
Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.
Idea 4 — DiagonalKVAttention defined ✓
  W_Q: full-rank  (d_model × d_head)
  W_K: proj + diagonal scale  (d_model×d_head + d_head params)
  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)
  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error
Idea 4b — GivensKVAttention defined ✓
  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale
  V recovery:     V = K * r  (u

## Combined: Ideas 1 + 2 + 4

**Stack:**
1. **Idea 4 (Diagonal KV):** `DiagonalKVAttention` — exact V recovery from K, 50% cache saving
2. **Idea 1 (XSA):** Applied during training from scratch — eliminates attention sinks,
   making Idea 2's PageRank eviction scores semantically meaningful
3. **Idea 2 (Keyword KV Cache):** PageRank-based token eviction on the already-halved K cache

**Combined memory:** Idea 4 halves KV → Idea 2 takes 34% of that → **~83% total reduction (6x)**

**PPL:** Diagonal KV heads must be trained from scratch. We train a small GPT-2-scale model
and compare perplexity to the full-rank baseline trained for the same number of steps.


In [41]:
# ── Combined Attention: Diagonal KV (Idea 4) + XSA (Idea 1) ──────────────────

class DiagonalKVXSAAttention(nn.Module):
    """
    DiagonalKVAttention + XSA projection (Idea 1).
    Intended for training from scratch — apply_xsa=True always here.

    XSA removes self-value component from output:
        z_i = y_i - (y_i · v̂_i) · v̂_i
    With diagonal V = K * r, the self-value vector is well-defined
    and the projection is as meaningful as in full-rank attention.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out    = d_out
        self.W_q      = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))
        self.w_v      = nn.Parameter(torch.ones(d_out))
        self.drop     = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x, token_ids=None, v_dedup=None):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k_proj(x) * self.w_k
        v = k * self.r

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        y    = attn @ v

        # XSA: remove self-value component
        v_norm = F.normalize(v, dim=-1)
        proj   = (y * v_norm).sum(dim=-1, keepdim=True)
        y      = y - proj * v_norm
        return y


class MHACombined(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVXSAAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, token_ids=None, v_dedup=None):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Combined attention (Diagonal KV + XSA) defined ✓")


Combined attention (Diagonal KV + XSA) defined ✓


In [42]:
# ── GPT shell — same structure as original, works with any MHA class ──────────

def _block_cls(cfg, mha_cls, mha_kwargs=None):
    kwargs = mha_kwargs or {}
    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = mha_cls(d, dh, cfg['context_length'],
                                cfg['drop'], cfg['n_head'], cfg['qkv_bias'], **kwargs)
            self.ffn  = FeedForward(cfg)
        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x), **kw))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x
    return Block


class GPT(nn.Module):
    def __init__(self, cfg, block_cls):
        super().__init__()
        self.cfg         = cfg
        d                = cfg['emb_dim']
        self.tok_emb     = nn.Embedding(cfg['vocab_size'], d)
        self.pos_emb     = nn.Embedding(cfg['context_length'], d)
        self.drop        = nn.Dropout(cfg['drop'])
        self.trfs_blocks = nn.ModuleList([block_cls() for _ in range(cfg['n_layers'])])
        self.final_norm  = LayerNorm(d)
        self.lm_head     = nn.Linear(d, cfg['vocab_size'], bias=False)

    def forward(self, idx, v_dedup=None):
        # FIX: always pass idx (token IDs) to blocks so Idea 3 dedup can fire.
        # Previously token_ids was gated and never reached the attention heads.
        if idx.dim() == 1: idx = idx.unsqueeze(0)
        B, T = idx.shape
        T    = min(T, self.cfg['context_length'])
        idx  = idx[:, -T:]
        x    = self.drop(self.tok_emb(idx)
                       + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.trfs_blocks:
            # Pass token_ids=idx unconditionally; MHA classes that don't accept it
            # will ignore it (baseline/XSA heads don't have the parameter so we
            # guard by checking if the att module supports it).
            try:
                x = blk(x, token_ids=idx, v_dedup=v_dedup)
            except TypeError:
                x = blk(x)
        return self.lm_head(self.final_norm(x))

print("GPT shell defined ✓")


GPT shell defined ✓


In [43]:
# ── Weight loaders ────────────────────────────────────────────────────────────
#
# load_gpt2_weights: loads into softmax baseline only (full-rank W_Q/K/V)
# init_diagonal_weights: random init for diagonal KV model (no pretrained weights)

def load_gpt2_weights(our_model, hf_model):
    """Load HuggingFace GPT-2 weights into the full-rank softmax baseline."""
    hf = hf_model.state_dict()
    sd = our_model.state_dict()

    sd['tok_emb.weight'].copy_(hf['wte.weight'])
    sd['pos_emb.weight'].copy_(hf['wpe.weight'])
    sd['final_norm.gamma'].copy_(hf['ln_f.weight'])
    sd['final_norm.beta' ].copy_(hf['ln_f.bias'  ])
    sd['lm_head.weight'  ].copy_(hf['wte.weight'])

    d, nh = our_model.cfg['emb_dim'], our_model.cfg['n_head']
    dh    = d // nh

    for i in range(our_model.cfg['n_layers']):
        p = f'h.{i}';  b = f'trfs_blocks.{i}'
        sd[f'{b}.ln1.gamma'].copy_(hf[f'{p}.ln_1.weight'])
        sd[f'{b}.ln1.beta' ].copy_(hf[f'{p}.ln_1.bias'  ])
        sd[f'{b}.ln2.gamma'].copy_(hf[f'{p}.ln_2.weight'])
        sd[f'{b}.ln2.beta' ].copy_(hf[f'{p}.ln_2.bias'  ])

        cw = hf[f'{p}.attn.c_attn.weight'].T
        cb = hf[f'{p}.attn.c_attn.bias'  ]
        Wq, Wk, Wv = cw[:d], cw[d:2*d], cw[2*d:]
        bq, bk, bv = cb[:d], cb[d:2*d], cb[2*d:]

        for h in range(nh):
            s, e = h*dh, (h+1)*dh
            hp   = f'{b}.att.heads.{h}'
            sd[f'{hp}.W_q.weight'].copy_(Wq[s:e])
            sd[f'{hp}.W_k.weight'].copy_(Wk[s:e])
            sd[f'{hp}.W_v.weight'].copy_(Wv[s:e])
            if our_model.cfg['qkv_bias']:
                sd[f'{hp}.W_q.bias'].copy_(bq[s:e])
                sd[f'{hp}.W_k.bias'].copy_(bk[s:e])
                sd[f'{hp}.W_v.bias'].copy_(bv[s:e])

        sd[f'{b}.att.out_proj.weight'].copy_(hf[f'{p}.attn.c_proj.weight'].T)
        sd[f'{b}.att.out_proj.bias'  ].copy_(hf[f'{p}.attn.c_proj.bias'  ])
        sd[f'{b}.ffn.layers.0.weight'].copy_(hf[f'{p}.mlp.c_fc.weight'   ].T)
        sd[f'{b}.ffn.layers.0.bias'  ].copy_(hf[f'{p}.mlp.c_fc.bias'     ])
        sd[f'{b}.ffn.layers.2.weight'].copy_(hf[f'{p}.mlp.c_proj.weight'  ].T)
        sd[f'{b}.ffn.layers.2.bias'  ].copy_(hf[f'{p}.mlp.c_proj.bias'   ])

    our_model.load_state_dict(sd)
    print("  ✓ Weights loaded")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("Weight loaders defined ✓")


Weight loaders defined ✓


In [44]:
# ── Perplexity evaluation ─────────────────────────────────────────────────────

@torch.no_grad()
def perplexity(model, token_ids, ctx, device, stride=128, use_dedup=False):
    """
    use_dedup: if True, enables the per-head VDeduplicator on combined_model heads.
               Dedup stores are reset at the start of each sliding window so that
               positional V vectors from one window don't bleed into the next.
               (Old behaviour: a single shared VDeduplicator was passed in from outside,
               causing cross-layer contamination and PPL=3258.  Fixed: each head owns
               its own store and it is reset per window.)
    """
    model.eval()
    token_ids = token_ids.to(device)
    T, nlls   = token_ids.size(1), []

    # Enable/disable per-head dedup
    if use_dedup:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'enable_dedup'):
                blk.att.enable_dedup()
    else:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'disable_dedup'):
                blk.att.disable_dedup()

    for begin in range(0, T - 1, stride):
        end  = min(begin + ctx, T)
        inp  = token_ids[:, begin:end-1]
        tgt  = token_ids[:, begin+1:end]
        if inp.size(1) == 0: break

        # Reset per-head V stores each window (avoid positional bleed across windows)
        if use_dedup:
            for blk in model.trfs_blocks:
                if hasattr(blk.att, 'reset_dedup'):
                    blk.att.reset_dedup()

        logits = model(inp, v_dedup=None)   # v_dedup arg ignored; heads are self-contained
        t      = logits.size(1)
        nlls.append(F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt[:, :t].reshape(-1)).item())

    # Leave dedup in whatever state caller wants after measurement
    return math.exp(sum(nlls) / len(nlls)), None


def get_text():
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    try:
        print("Fetching the-verdict.txt ...")
        with urllib.request.urlopen(url, timeout=15) as r:
            text = r.read().decode('utf-8')
        print(f"  Got {len(text):,} chars")
        return text
    except Exception as e:
        print(f"  Download failed ({e}), using fallback.")
        return "Every effort moves you forward. " * 500

print("Perplexity fn defined ✓")
print("  Fix: per-head dedup reset each window (no positional bleed)")


Perplexity fn defined ✓
  Fix: per-head dedup reset each window (no positional bleed)


In [45]:
# ── MAIN RUN ──────────────────────────────────────────────────────────────────

def run():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice : {device}")

    cfg = {
        'vocab_size':     50257,
        'n_head':         12,
        'drop':           0.1,
        'n_layers':       12,
        'context_length': 1024,
        'qkv_bias':       True,
        'emb_dim':        768,
    }
    dh = cfg['emb_dim'] // cfg['n_head']

    text      = get_text()
    enc       = tiktoken.get_encoding("gpt2")
    all_ids   = enc.encode(text, allowed_special={'<|endoftext|>'})
    token_ids = torch.tensor(all_ids, dtype=torch.long).unsqueeze(0)
    print(f"Tokens  : {token_ids.size(1):,}\n")

    # ── [1] Softmax baseline — load pretrained GPT-2 ──────────────────────
    print("[1/3] Softmax baseline (pretrained GPT-2) ...")
    from transformers import GPT2Model
    hf_model = GPT2Model.from_pretrained('gpt2')
    hf_model.eval()
    softmax_model = GPT(cfg, _block_cls(cfg, MHASoftmax)).to(device)
    load_gpt2_weights(softmax_model, hf_model)
    ppl_base, _ = perplexity(softmax_model, token_ids, cfg['context_length'], device)
    print(f"  Pretrained GPT-2 PPL = {ppl_base:.2f}\n")

    # ── [2] Diagonal KV model — train from scratch ─────────────────────────
    print("[2/3] Diagonal KV model (train from scratch) ...")
    diag_model = GPT(cfg, _block_cls(cfg, MHADiagonalKV)).to(device)
    n_total, n_train = count_params(diag_model)
    n_base,  _       = count_params(softmax_model)
    print(f"  Diagonal KV params : {n_total:,}")
    print(f"  Softmax baseline   : {n_base:,}")
    print(f"  Reduction          : {(1 - n_total/n_base)*100:.1f}% fewer params")

    # Quick training loop on the-verdict corpus
    # Note: a real comparison needs far more data and steps — this demonstrates
    # the architecture trains and converges, not that it matches GPT-2 quality.
    TRAIN_STEPS = 500
    SEQ_LEN     = 128
    LR          = 3e-4
    optimizer = torch.optim.AdamW(diag_model.parameters(), lr=LR, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_STEPS, eta_min=LR*0.1)

    T_total = token_ids.size(1)
    diag_model.train()
    losses = []
    print(f"  Training {TRAIN_STEPS} steps, seq_len={SEQ_LEN}, lr={LR} ...")
    for step in range(TRAIN_STEPS):
        start = torch.randint(0, max(1, T_total - SEQ_LEN - 1), (1,)).item()
        inp   = token_ids[:, start:start+SEQ_LEN].to(device)
        tgt   = token_ids[:, start+1:start+SEQ_LEN+1].to(device)
        logits = diag_model(inp)
        loss   = F.cross_entropy(logits.reshape(-1, cfg['vocab_size']),
                                 tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diag_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
        if (step+1) % 100 == 0:
            avg = sum(losses[-100:])/100
            print(f"    Step {step+1:4d}/{TRAIN_STEPS}  loss={avg:.4f}  ppl={math.exp(avg):.1f}  lr={scheduler.get_last_lr()[0]:.2e}")

    diag_model.eval()
    ppl_diag, _ = perplexity(diag_model, token_ids, cfg['context_length'], device)
    print(f"  Diagonal KV PPL (after {TRAIN_STEPS} steps) = {ppl_diag:.2f}\n")

    # ── [3] Combined: Diagonal KV + XSA + PageRank cache ──────────────────
    print("[3/3] Combined model (Diagonal KV + XSA + PageRank cache) ...")
    combined_model = GPT(cfg, _block_cls(cfg, MHACombined)).to(device)

    TRAIN_STEPS_C = 500
    optimizer_c = torch.optim.AdamW(combined_model.parameters(), lr=LR, weight_decay=0.1)
    scheduler_c = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=TRAIN_STEPS_C, eta_min=LR*0.1)
    combined_model.train()
    losses_c = []
    print(f"  Training {TRAIN_STEPS_C} steps ...")
    for step in range(TRAIN_STEPS_C):
        start  = torch.randint(0, max(1, T_total - SEQ_LEN - 1), (1,)).item()
        inp    = token_ids[:, start:start+SEQ_LEN].to(device)
        tgt    = token_ids[:, start+1:start+SEQ_LEN+1].to(device)
        logits = combined_model(inp)
        loss   = F.cross_entropy(logits.reshape(-1, cfg['vocab_size']), tgt.reshape(-1))
        optimizer_c.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(combined_model.parameters(), 1.0)
        optimizer_c.step()
        scheduler_c.step()
        losses_c.append(loss.item())
        if (step+1) % 100 == 0:
            avg = sum(losses_c[-100:])/100
            print(f"    Step {step+1:4d}/{TRAIN_STEPS_C}  loss={avg:.4f}  ppl={math.exp(avg):.1f}  lr={scheduler_c.get_last_lr()[0]:.2e}")

    combined_model.eval()
    ppl_combined, _ = perplexity(combined_model, token_ids, cfg['context_length'], device)

    # ── Memory analysis ────────────────────────────────────────────────────
    n        = token_ids.size(1)
    nl, nh   = cfg['n_layers'], cfg['n_head']
    std_mb   = 2 * nl * nh * n * dh * 2 / 1e6          # K + V, float16
    diag_mb  = nl * nh * n * dh * 2 / 1e6              # K only (V recovered from r)
    kw_n     = int(KEYWORD_WINDOW + (n - KEYWORD_WINDOW) * KEYWORD_RATIO)
    comb_mb  = nl * nh * kw_n * dh * 2 / 1e6           # K only + PageRank eviction

    print(f"\n{'='*60}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"  [Baseline]   Softmax (pretrained GPT-2):     PPL = {ppl_base:.2f}")
    print(f"  [Idea 4]     Diagonal KV ({TRAIN_STEPS} steps):       PPL = {ppl_diag:.2f}")
    print(f"  [Ideas 1+2+4] Combined ({TRAIN_STEPS_C} steps):       PPL = {ppl_combined:.2f}")
    print()
    print(f"  KV Cache Memory ({n:,} tokens, fp16):")
    print(f"    Standard (K+V):              {std_mb:.1f} MB")
    print(f"    Diagonal KV (K only):        {diag_mb:.1f} MB  (-50%, exact)")
    print(f"    + PageRank eviction:         {comb_mb:.1f} MB  (-{(1-comb_mb/std_mb)*100:.0f}% total, ~{std_mb/comb_mb:.1f}x)")
    print()
    print(f"  Note: PPL comparison is indicative only — diagonal model trained")
    print(f"  on {T_total} tokens vs GPT-2's ~10B tokens. Architecture is sound;")
    print(f"  quality gap closes with scale.")
    print(f"{'='*60}")

    # ── XSA sink analysis on combined model ───────────────────────────────
    print(f"\nXSA Attention Sink Analysis (combined model, layer 0, head 0):")
    dummy_ids = token_ids[:, :64].to(device)
    x_emb = combined_model.drop(
        combined_model.tok_emb(dummy_ids)
        + combined_model.pos_emb(torch.arange(64, device=device)))
    with torch.no_grad():
        h0  = combined_model.trfs_blocks[0].att.heads[0]
        k   = h0.W_k_proj(x_emb[0]) * h0.w_k
        v   = k * h0.r
        q   = h0.W_q(x_emb[0])
        s   = (q @ k.T) / math.sqrt(dh)
        s   = s.masked_fill(torch.triu(torch.ones(64,64,device=device),diagonal=1).bool(), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        y    = attn @ v
        v_norm = F.normalize(v, dim=-1)
        y_xsa  = y - (y * v_norm).sum(-1, keepdim=True) * v_norm
        sim_before = F.cosine_similarity(y, v, dim=-1).mean().item()
        sim_after  = F.cosine_similarity(y_xsa, v, dim=-1).mean().item()
    print(f"  Cosine(output, self_value) before XSA : {sim_before:.4f}")
    print(f"  Cosine(output, self_value) after  XSA : {sim_after:.4f}  ← should be ~0")
    print(f"  Attention sink eliminated: {'✓' if abs(sim_after) < 0.1 else '✗'}")

    # ── r ratio analysis ──────────────────────────────────────────────────
    print(f"\nDiagonal KV ratio (r = w_v/w_k) analysis — diag model, layer 0:")
    with torch.no_grad():
        for hi in range(min(4, nh)):
            h = diag_model.trfs_blocks[0].att.heads[hi]
            r = h.r
            print(f"  Head {hi}: r mean={r.mean():.3f}  std={r.std():.3f}  "
                  f"min={r.min():.3f}  max={r.max():.3f}")

    return softmax_model, diag_model, combined_model, token_ids, cfg, device


softmax_model, diag_model, combined_model, token_ids, cfg, device = run()



Device : cuda
Fetching the-verdict.txt ...
  Got 20,479 chars
Tokens  : 5,145

[1/3] Softmax baseline (pretrained GPT-2) ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Weights loaded
  Pretrained GPT-2 PPL = 35.83

[2/3] Diagonal KV model (train from scratch) ...
  Diagonal KV params : 155,959,296
  Softmax baseline   : 163,037,184
  Reduction          : 4.3% fewer params
  Training 500 steps, seq_len=128, lr=0.0003 ...
    Step  100/500  loss=6.3449  ppl=569.6  lr=2.74e-04
    Step  200/500  loss=3.9779  ppl=53.4  lr=2.07e-04
    Step  300/500  loss=3.0607  ppl=21.3  lr=1.23e-04
    Step  400/500  loss=2.5730  ppl=13.1  lr=5.58e-05
    Step  500/500  loss=2.3550  ppl=10.5  lr=3.00e-05
  Diagonal KV PPL (after 500 steps) = 14.11

[3/3] Combined model (Diagonal KV + XSA + PageRank cache) ...
  Training 500 steps ...
    Step  100/500  loss=6.2867  ppl=537.4  lr=2.74e-04
    Step  200/500  loss=3.8832  ppl=48.6  lr=2.07e-04
    Step  300/500  loss=3.0992  ppl=22.2  lr=1.23e-04
    Step  400/500  loss=2.4726  ppl=11.9  lr=5.58e-05
    Step  500/500  loss=2.2725  ppl=9.7  lr=3.00e-05

RESULTS SUMMARY
  [Baseline]   Softmax (pretrained GPT-2):     PPL

### GPU Memory Cleanup

Run this cell **before** the eval cell below.
The `run()` call above left three full GPT-2-scale models on GPU (~13 GB).
This frees them so the eval cell has room to train new models.

In [46]:
# ── GPU Memory Cleanup — run this before the eval cell ───────────────────────
# The three models from run() (softmax_model, diag_model, combined_model)
# are still allocated on GPU and consuming ~13 GB.
# Delete them explicitly before training new models.

import gc, torch

_models_to_free = ["softmax_model", "diag_model", "combined_model",
                   "_softmax_full"]   # also freed if eval cell partially ran

for _name in _models_to_free:
    if _name in dir():
        try:
            _m = eval(_name)
            _m.cpu()           # move weights off GPU first
            del _m
        except Exception:
            pass

# Also clear any lingering HuggingFace model
try:
    hf_model.cpu()
    del hf_model
except Exception:
    pass
try:
    _hf.cpu()
    del _hf
except Exception:
    pass

# Force Python GC then empty PyTorch cache
gc.collect()
torch.cuda.empty_cache()

# Report free memory
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory after cleanup:")
    print(f"  Free  : {free/1e9:.2f} GB")
    print(f"  Total : {total/1e9:.2f} GB")
    print(f"  In use: {(total-free)/1e9:.2f} GB")
else:
    print("No CUDA device found.")

print()
print("Safe to run the eval cell now.")


GPU memory after cleanup:
  Free  : 11.59 GB
  Total : 15.64 GB
  In use: 4.05 GB

Safe to run the eval cell now.


## Proper Evaluation — Train/Test Split + Diagnostic Metrics

**Why the previous numbers were misleading:**
All models were evaluated on the same tokens they trained on.
With only ~5 K tokens and 500 steps the model memorises the text, so
train-PPL collapses while test-PPL (generalisation) remains high.

**What this cell fixes:**

1. **80/20 train/test split** — first 80 % of tokens for training,
   last 20 % held out and never seen during training.
   All PPL figures below are on the **held-out test set only**.

2. **Train-loss curve** — logged every 50 steps so you can see whether
   the model is still learning or has saturated.

3. **Train/test PPL gap** — quantifies overfitting.
   A gap > 5× means the model memorised; gap < 2× means reasonable generalisation.

4. **Bits-per-character (BPC)** — `PPL^(1/avg_chars_per_token)`, usually
   `PPL^(1/4)` for BPE. Comparable to character-level LM benchmarks.

5. **Token-level accuracy** — fraction of next-token predictions where
   argmax == target. Complementary to PPL; less sensitive to tail distributions.

6. **Calibration check** — mean softmax confidence on correct token.
   High confidence + low accuracy = overconfident/memorised.
   Low confidence + reasonable accuracy = well-calibrated generalisation.

7. **Givens sweep on TRAIN tokens only, evaluated on TEST tokens** —
   the only comparison that tells you whether more rotation helps generalisation
   rather than just memorisation.


In [48]:
# ── Definitions inlined for standalone execution ────────────────────────────
import math, urllib.request, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from transformers import GPT2Model
import rustworkx as rx           # ← Rust-backed graph library for PageRank

warnings.filterwarnings('ignore')
torch.manual_seed(42)

MAX_TOKENS       = 4096
KEYWORD_RATIO    = 0.3   # Idea 2: keep top 30% of tokens after threshold
KEYWORD_WINDOW   = 256   # Idea 2: tokens before this = full cache; after = keyword only
DEDUP_ENABLED    = True  # Idea 3: deduplicate V by token id
EIGENBASIS_RANK  = 32    # Idea 4: rank-r approx of shared eigenbasis (None = disabled)

# ── Shared layers (unchanged from original) ───────────────────────────────────

class LayerNorm(nn.Module):
    """Fixed: unbiased=False variance matches GPT-2 paper."""
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta  = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        return (x - mean) / (var + self.eps).sqrt() * self.gamma + self.beta

class GeLU(nn.Module):
    def forward(self, x): return F.gelu(x)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['emb_dim']
        self.layers = nn.Sequential(nn.Linear(d, 4*d), GeLU(), nn.Linear(4*d, d))
    def forward(self, x): return self.layers(x)


# ── Baseline: standard softmax attention (original notebook) ──────────────────

class CausalAttentionSoftmax(nn.Module):
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        return self.drop(torch.softmax(s, dim=-1)) @ v

class MHASoftmax(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionSoftmax(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)
    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))


# ── Idea 1: XSA — Exclusive Self-Attention ────────────────────────────────────
# Zhai, Apple 2026 (arXiv:2603.09078)
#
# IMPORTANT: The XSA projection is learned behaviour — it must be trained in
# from scratch.  Applying it post-hoc to GPT-2 pretrained weights removes signal
# the model already learnt to rely on, so PPL rises.
# We therefore use apply_xsa=False for perplexity measurement and apply_xsa=True
# only for the attention-sink analysis where we demonstrate the projection works.

class CausalAttentionXSA(nn.Module):
    """
    Standard causal attention + XSA projection step.
    After computing y = softmax(QK^T/√d)·V, we project out the
    component of y along the token's own value vector v_i:

        z_i = y_i - (y_i · v̂_i) · v̂_i

    This eliminates attention similarity bias / attention sinks.
    Use apply_xsa=False at inference on pretrained weights (PPL measurement).
    Use apply_xsa=True only when the model was trained with XSA from scratch.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.W_q  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.apply_xsa = False   # ← set True only when trained from scratch

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        s = (q @ k.transpose(1,2)) / math.sqrt(k.shape[-1])
        s = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = self.drop(torch.softmax(s, dim=-1))
        y = attn @ v

        if self.apply_xsa:
            # Remove self-value projection: z_i = y_i - (y_i·v̂_i)·v̂_i
            v_norm = F.normalize(v, dim=-1)
            proj   = (y * v_norm).sum(dim=-1, keepdim=True)
            y      = y - proj * v_norm

        return y

class MHAXSA(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionXSA(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def set_apply_xsa(self, flag: bool):
        for h in self.heads:
            h.apply_xsa = flag

    def forward(self, x):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]")


# ── Idea 2: Keyword Threshold KV Cache — rustworkx PageRank ──────────────────
#
# KEY INSIGHT: Build a directed attention graph for the flush batch using
# rustworkx.PyDiGraph.  Edge i→j carries weight = attention_weight[i,j].
# Run rustworkx.pagerank() — tokens that many others attend to get high score.
# Keep top KEYWORD_RATIO fraction by PageRank score; discard the rest.
#
# rustworkx is a Rust-backed Python graph library (used in Qiskit).
# It is significantly faster than networkx for this workload.

import rustworkx as rx

class KeywordKVCache:
    """
    Manages a two-zone KV cache with rustworkx PageRank importance scoring.

      Zone A (recent): last `window` tokens → full exact K, V stored.
      Zone B (old):    tokens beyond window → only top-k% 'keywords' kept.

    Importance scoring (flush time):
      1. Build PyDiGraph: node per token, edge i→j weighted by attn_weight[i,j].
      2. Run rustworkx.pagerank(G, alpha=0.85, weight_fn=lambda e: e).
      3. Rank nodes by PageRank score; keep top KEYWORD_RATIO fraction.

    Usage:
      cache = KeywordKVCache(window=256, ratio=0.3, d_head=64, device=device)
      cache.update(k_vec, v_vec, attn_row)   # one token at a time
      K, V  = cache.get_kv()                 # retrieve full usable KV tensors
      stats = cache.memory_stats()
    """

    def __init__(self, window: int, ratio: float, d_head: int, device):
        self.window  = window
        self.ratio   = ratio
        self.d_head  = d_head
        self.device  = device

        # Zone A — recent tokens (full K, V stored)
        self.recent_K    = []   # list of [d_head] CPU tensors
        self.recent_V    = []
        # attn_rows[t] = the attention ROW for token t:
        #   attn_rows[t][j] = weight that token t placed on token j (j ≤ t).
        # This is the row of the attention matrix, NOT the column.
        self.recent_attn_rows = []   # list of 1-D tensors, variable length

        # Zone B — evicted keyword tokens
        self.keyword_K = []
        self.keyword_V = []

        self.total_tokens       = 0
        self.pagerank_calls     = 0
        self.last_pr_scores     = None   # PageRank scores from last flush (for inspection)

    # ------------------------------------------------------------------
    def update(self, k: torch.Tensor, v: torch.Tensor,
               attn_row: torch.Tensor = None):
        """
        Add one token's K and V to the cache.

        attn_row: 1-D tensor of length (t+1) giving the attention weights
                  that THIS token placed on tokens 0..t-1 (i.e. row t of the
                  causal attention matrix: attn[t, 0..t]).
                  If None, a uniform weight of 1/n is assumed (degrades to
                  uniform PageRank, still correct).
        """
        self.recent_K.append(k.detach().cpu())
        self.recent_V.append(v.detach().cpu())

        n = len(self.recent_K)
        if attn_row is not None:
            row = attn_row.detach().cpu()
        else:
            row = torch.full((n,), 1.0 / n)

        self.recent_attn_rows.append(row)
        self.total_tokens += 1

        if len(self.recent_K) > self.window:
            self._flush_with_pagerank()

    # ------------------------------------------------------------------
    def _build_attention_graph(self, attn_rows, n):
        """
        Build a rustworkx PyDiGraph from the causal attention matrix.

        attn_rows[src] is the ATTENTION ROW for token `src`:
          attn_rows[src][dst] = weight that token `src` placed on token `dst`.
        Causal constraint: dst <= src (future tokens are not attended to).

        Edge direction: src → dst  ("token src attends to token dst")
        Weight = attention_weight[src, dst].

        PageRank semantics: a node (token) with many high-weight INCOMING edges
        is one that many tokens attend to — exactly the tokens worth keeping.

        Returns: (PyDiGraph, list of node indices)
        """
        G = rx.PyDiGraph()
        node_ids = G.add_nodes_from(list(range(n)))

        edges = []
        for src in range(n):
            row = attn_rows[src]                      # weights src placed on 0..src
            dst_len = min(len(row), src + 1)          # causal: dst ≤ src
            for dst in range(dst_len):
                w = float(row[dst].item())
                if w > 1e-4:                          # threshold (1e-4 not 1e-6)
                    edges.append((src, dst, w))       # src attends to dst

        if edges:
            G.add_edges_from(edges)

        return G, node_ids

    # ------------------------------------------------------------------
    def _flush_with_pagerank(self):
        """
        Move the oldest half of the recent zone to the keyword zone.

        Steps:
          1. Take oldest `flush_n` tokens.
          2. Build rustworkx PyDiGraph with attention weights.
          3. Run PageRank to get importance scores.
          4. Keep top KEYWORD_RATIO fraction; discard the rest.
        """
        flush_n = self.window // 2
        flush_K    = self.recent_K[:flush_n]
        flush_V    = self.recent_V[:flush_n]
        flush_rows = self.recent_attn_rows[:flush_n]

        # ── Build attention graph ──────────────────────────────────────
        G, node_ids = self._build_attention_graph(flush_rows, flush_n)

        # ── PageRank on the attention graph ───────────────────────────
        # weight_fn maps edge payload (float) → float weight for PageRank
        try:
            pr_map = rx.pagerank(G, alpha=0.85, weight_fn=lambda e: float(e))
        except Exception:
            # Fallback: if graph has no edges, uniform scores
            pr_map = {i: 1.0 / flush_n for i in range(flush_n)}

        self.pagerank_calls += 1
        pr_keys   = set(pr_map.keys())
        pr_scores = [pr_map[i] if i in pr_keys else 0.0 for i in range(flush_n)]
        self.last_pr_scores = pr_scores   # store for inspection

        # ── Select top-k% by PageRank ─────────────────────────────────
        keep_n   = max(1, int(flush_n * self.ratio))
        ranked   = sorted(range(flush_n), key=lambda i: pr_scores[i], reverse=True)
        keep_idx = set(ranked[:keep_n])

        for i in range(flush_n):
            if i in keep_idx:
                self.keyword_K.append(flush_K[i])
                self.keyword_V.append(flush_V[i])
            # else: discarded — true memory saving

        # ── Trim recent zone ──────────────────────────────────────────
        self.recent_K         = self.recent_K[flush_n:]
        self.recent_V         = self.recent_V[flush_n:]
        self.recent_attn_rows = self.recent_attn_rows[flush_n:]

    # ------------------------------------------------------------------
    def get_kv(self):
        """Return full usable K, V tensors: keywords + recent. [n_kept, d_head]"""
        all_K = self.keyword_K + self.recent_K
        all_V = self.keyword_V + self.recent_V
        if not all_K:
            return None, None
        K = torch.stack(all_K).to(self.device)
        V = torch.stack(all_V).to(self.device)
        return K, V

    # ------------------------------------------------------------------
    def memory_stats(self):
        full_n   = self.total_tokens
        stored_n = len(self.keyword_K) + len(self.recent_K)
        saved    = (1 - stored_n / max(full_n, 1)) * 100
        return {
            'total':          full_n,
            'stored':         stored_n,
            'keywords':       len(self.keyword_K),
            'recent':         len(self.recent_K),
            'saved_pct':      saved,
            'pagerank_calls': self.pagerank_calls,
        }


# ── Quick smoke test ──────────────────────────────────────────────────────────
def _smoke_test_keyword_cache():
    device = torch.device('cpu')
    d      = 64
    cache  = KeywordKVCache(window=16, ratio=0.3, d_head=d, device=device)

    torch.manual_seed(0)
    # Simulate 40 tokens with random K/V and random attention columns
    for t in range(40):
        k = torch.randn(d)
        v = torch.randn(d)
        # Simulate a causal attention column for this token
        col = torch.softmax(torch.randn(t + 1), dim=0)
        cache.update(k, v, attn_row=col)

    K, V = cache.get_kv()
    stats = cache.memory_stats()
    print(f"Smoke test: 40 tokens → stored={stats['stored']} "
          f"(keywords={stats['keywords']}, recent={stats['recent']}), "
          f"saved={stats['saved_pct']:.0f}%, "
          f"PageRank calls={stats['pagerank_calls']}")
    print(f"  K shape: {K.shape}, V shape: {V.shape}")
    print(f"  Last PageRank scores (first 8): "
          f"{[f'{s:.4f}' for s in (cache.last_pr_scores or [])[:8]]}")
    assert K.shape[0] == stats['stored'], "K shape mismatch"
    print("  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed")

_smoke_test_keyword_cache()
print("\nIdea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓")

# ── Idea 3: Retired — stub only ──────────────────────────────────────────────
# Works on RoPE models only. See markdown above for explanation.

class VDeduplicator:
    """Stub — no-op for GPT-2. Exact implementation valid for LLaMA/Mistral."""
    def __init__(self): pass
    def get_or_store(self, token_id, v): return v

print("Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.")


# ── Idea 4: Diagonal W_K / W_V Attention ─────────────────────────────────────
#
# W_Q: full-rank nn.Linear (d_model → d_head)  — unchanged, Q not cached
# W_K: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_k)
# W_V: diagonal — nn.Parameter vector [d_head], applied as x @ diag(w_v)
#   but x is [B, T, d_model] and d_model != d_head, so we need a projection.
#   Solution: W_K_proj: Linear(d_model → d_head, bias=False) THEN diagonal scale.
#   This keeps the input projection (for dimensionality reduction) but the
#   "rotation" in head space is replaced by a diagonal scale.
#
# Memory at inference:
#   Standard: cache K [n, d_head] + V [n, d_head] = 2n*d_head values
#   Diagonal: cache K [n, d_head] + r [d_head]    = n*d_head + d_head values
#   Saving:   ~50% (r is negligible vs n*d_head for long contexts)
#
# V recovery: V = K * r  where r = w_v / w_k  (element-wise, broadcast over n)
#   This is EXACT — no approximation, no fine-tuning needed.

class DiagonalKVAttention(nn.Module):
    """
    Causal attention with diagonal W_K and W_V in head space.

    Architecture:
      Q = x @ W_Q^T                    (full-rank, d_model → d_head)
      K = (x @ W_K_proj^T) * w_k       (project then diagonal-scale)
      V = K * r    where r = w_v/w_k   (exact recovery, no storage needed)

    KV cache at inference:
      Store: K [n, d_head]  +  r [d_head]  (shared across all n tokens)
      Recover: V = K * r  (one elementwise multiply, O(n*d_head))
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank projection (not cached — no reason to restrict)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: linear projection into head space, then diagonal scale
        # W_K_proj reduces d_model → d_head (the expensive part, kept for expressivity)
        # w_k is the diagonal scale applied after projection (d_head params)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))   # diagonal K scale

        # V: diagonal scale only — w_v / w_k gives the recovery ratio r
        self.w_v      = nn.Parameter(torch.ones(d_out))   # diagonal V scale

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        # Initialise w_k and w_v with small random values (like default Linear init)
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x)                          # [B, T, d_head]  full-rank
        k = self.W_k_proj(x) * self.w_k          # [B, T, d_head]  diagonal-scaled
        v = k * self.r                            # [B, T, d_head]  exact recovery

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        """Report memory usage vs standard attention."""
        return {'r': self.r.detach().cpu(),
                'w_k_range': (self.w_k.min().item(), self.w_k.max().item()),
                'w_v_range': (self.w_v.min().item(), self.w_v.max().item())}


class MHADiagonalKV(nn.Module):
    """Multi-head attention with diagonal K/V and full-rank Q."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass   # XSA not used on diagonal heads (trained from scratch)

print("Idea 4 — DiagonalKVAttention defined ✓")
print(f"  W_Q: full-rank  (d_model × d_head)")
print(f"  W_K: proj + diagonal scale  (d_model×d_head + d_head params)")
print(f"  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)")
print(f"  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error")


# ── Idea 4b: GivensRotation + DiagonalKV ─────────────────────────────────────
#
# A GivensRotation module applies p sequential 2D rotations in learned (i,j) planes.
# Each rotation is parameterised by one learned angle θ (initialised to 0 → identity).
#
# Architecture per head:
#   x_proj = W_K_proj(x)               [B, T, d_head]  full-rank (shared)
#   x_rot  = GivensRotation(x_proj)    [B, T, d_head]  p learned rotations
#   K      = x_rot * w_k               [B, T, d_head]  diagonal scale
#   V      = K * r                     [B, T, d_head]  exact recovery (r = w_v/w_k)
#
# V recovery proof:
#   K = diag(w_k) @ R @ x_proj
#   V = diag(w_v) @ R @ x_proj  (same R)
#     = diag(w_v/w_k) @ K = K * r   ✓
#
# n_givens_pairs controls expressivity:
#   0    → pure diagonal (Idea 4 baseline)
#   d//2 → one half-sweep (all pairs once, non-overlapping)
#   d    → one full sweep (recommended default, 64 params for d_head=64)
#   d*(d-1)//2 → full rotation group (equivalent to full-rank W_K, factored)

import itertools

class GivensRotation(nn.Module):
    """
    Learnable product of p Givens (plane) rotations applied to the last dimension.

    Each rotation acts in a 2D subspace (axes i, j) by angle θ:
        x[..., i] ← x[..., i]*cos(θ) - x[..., j]*sin(θ)
        x[..., j] ← x[..., i]*sin(θ) + x[..., j]*cos(θ)

    The p (i, j) plane pairs are fixed at construction (sequential sweep of consecutive
    pairs: (0,1), (2,3), ..., then (1,2), (3,4), ... for p > d//2).
    Only the angles θ are learned.

    Initialisation: θ = 0  →  identity at the start of training.
    The model starts as a pure diagonal model and grows rotational structure as needed.

    Args:
        d_head        : dimension being rotated
        n_pairs       : number of Givens rotations (p). 0 = identity (no-op).
    """
    def __init__(self, d_head: int, n_pairs: int):
        super().__init__()
        self.d_head  = d_head
        self.n_pairs = n_pairs

        if n_pairs == 0:
            # No-op: register no parameters, forward is an identity.
            self.register_buffer('_dummy', torch.zeros(1))
            return

        # ── Generate plane pairs ──────────────────────────────────────────
        # Sweep consecutive pairs in multiple passes until we have n_pairs.
        # Pass 0: (0,1), (2,3), (4,5), ...         (non-overlapping, parallelisable)
        # Pass 1: (1,2), (3,4), (5,6), ...
        # Pass 2: (0,1), (2,3), ...  (repeat)
        # This covers all adjacent pairs before reusing any, giving diverse mixing.
        planes = []
        pass_idx = 0
        while len(planes) < n_pairs:
            offset = pass_idx % 2          # alternates 0 and 1
            start  = offset
            for i in range(start, d_head - 1, 2):
                planes.append((i, i + 1))
                if len(planes) == n_pairs:
                    break
            pass_idx += 1

        self.planes = planes               # list of (i, j) tuples, length n_pairs

        # Learned angles — initialised to 0 (identity rotation)
        self.angles = nn.Parameter(torch.zeros(n_pairs))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [..., d_head]
        Returns rotated tensor of same shape.
        """
        if self.n_pairs == 0:
            return x

        # Clone to avoid in-place autograd issues across loop iterations
        x = x.clone()

        for k, (i, j) in enumerate(self.planes):
            theta = self.angles[k]
            cos_t = torch.cos(theta)
            sin_t = torch.sin(theta)
            xi = x[..., i].clone()
            xj = x[..., j].clone()
            x[..., i] = cos_t * xi - sin_t * xj
            x[..., j] = sin_t * xi + cos_t * xj

        return x

    def extra_repr(self) -> str:
        return f"d_head={self.d_head}, n_pairs={self.n_pairs}"


# ── GivensKVAttention — drop-in replacement for DiagonalKVAttention ───────────

class GivensKVAttention(nn.Module):
    """
    DiagonalKVAttention augmented with a shared learnable Givens rotation.

    Architecture:
      Q      = W_Q(x)                              full-rank (d_model → d_head)
      x_proj = W_K_proj(x)                         shared projection
      x_rot  = GivensRotation(x_proj)              p learned rotations
      K      = x_rot * w_k                         diagonal scale
      V      = K * r   where r = w_v / w_k         exact recovery ✓

    KV cache at inference:
      Store K [n, d_head]  +  r [d_head]   →  same 50% saving as Idea 4.
      Reconstruction: V = K * r  (unchanged from Idea 4).

    Args:
      d_in, d_out   : as per DiagonalKVAttention
      n_givens_pairs: number of Givens rotations (0 = pure diagonal baseline)
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.d_out = d_out

        # Q: full-rank (not cached)
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)

        # K: project, rotate (Givens), diagonal scale
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.givens   = GivensRotation(d_out, n_givens_pairs)
        self.w_k      = nn.Parameter(torch.ones(d_out))

        # V: exact recovery from K
        self.w_v      = nn.Parameter(torch.ones(d_out))

        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None

        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)
        # angles initialised to 0 inside GivensRotation → identity start

    @property
    def r(self):
        """Recovery ratio: V = K * r.  Clamped to avoid explosion at w_k ≈ 0."""
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x):
        B, T, _ = x.shape

        q      = self.W_q(x)                        # [B, T, d_head]
        x_proj = self.W_k_proj(x)                   # [B, T, d_head]  shared projection
        x_rot  = self.givens(x_proj)                # [B, T, d_head]  Givens rotation
        k      = x_rot * self.w_k                   # [B, T, d_head]  diagonal scale
        v      = k * self.r                         # [B, T, d_head]  exact V recovery

        s    = (q @ k.transpose(1, 2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T, :T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        return attn @ v

    def kv_cache_stats(self):
        return {
            'r':           self.r.detach().cpu(),
            'w_k_range':   (self.w_k.min().item(), self.w_k.max().item()),
            'w_v_range':   (self.w_v.min().item(), self.w_v.max().item()),
            'angles_std':  self.givens.angles.std().item() if self.givens.n_pairs > 0 else 0.0,
        }


class MHAGivensKV(nn.Module):
    """Multi-head attention using GivensKVAttention heads."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads,
                 qkv_bias=False, n_givens_pairs=64):
        super().__init__()
        self.heads = nn.ModuleList([
            GivensKVAttention(d_model, d_head, ctx, drop, qkv_bias,
                              n_givens_pairs=n_givens_pairs)
            for _ in range(n_heads)
        ])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

    def set_apply_xsa(self, flag): pass  # XSA not used here


def make_givens_block_cls(cfg, n_givens_pairs=64):
    """Factory: returns a Block class with GivensKV attention."""
    d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']

    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = MHAGivensKV(d, dh, cfg['context_length'], cfg['drop'],
                                    cfg['n_head'], cfg['qkv_bias'],
                                    n_givens_pairs=n_givens_pairs)
            self.ffn  = FeedForward(cfg)

        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x)))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x

    return Block


print("Idea 4b — GivensKVAttention defined ✓")
print(f"  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale")
print(f"  V recovery:     V = K * r  (unchanged — exact, zero approximation error)")
print(f"  KV cache:       store K only + r per head  →  50% saving preserved")
print(f"  Init:           angles=0  →  identity at t=0  (pure diagonal start)")
print()
print("Plane generation for d_head=64, n_pairs=8:")
_demo = GivensRotation(64, 8)
print(f"  planes = {_demo.planes}")
print(f"  angles = {_demo.angles.data.tolist()}  (all zeros → identity)")


# ── Combined Attention: Diagonal KV (Idea 4) + XSA (Idea 1) ──────────────────

class DiagonalKVXSAAttention(nn.Module):
    """
    DiagonalKVAttention + XSA projection (Idea 1).
    Intended for training from scratch — apply_xsa=True always here.

    XSA removes self-value component from output:
        z_i = y_i - (y_i · v̂_i) · v̂_i
    With diagonal V = K * r, the self-value vector is well-defined
    and the projection is as meaningful as in full-rank attention.
    """
    def __init__(self, d_in, d_out, ctx, drop, qkv_bias=False):
        super().__init__()
        self.d_out    = d_out
        self.W_q      = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k_proj = nn.Linear(d_in, d_out, bias=False)
        self.w_k      = nn.Parameter(torch.ones(d_out))
        self.w_v      = nn.Parameter(torch.ones(d_out))
        self.drop     = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1))
        self.last_attn_weights = None
        nn.init.normal_(self.w_k, mean=1.0, std=0.02)
        nn.init.normal_(self.w_v, mean=1.0, std=0.02)

    @property
    def r(self):
        return self.w_v / self.w_k.clamp(min=1e-6)

    def forward(self, x, token_ids=None, v_dedup=None):
        B, T, _ = x.shape
        q = self.W_q(x)
        k = self.W_k_proj(x) * self.w_k
        v = k * self.r

        s    = (q @ k.transpose(1,2)) / math.sqrt(self.d_out)
        s    = s.masked_fill(self.mask[:T,:T].bool().unsqueeze(0), float('-inf'))
        attn = torch.softmax(s, dim=-1)
        self.last_attn_weights = attn.detach()
        attn = self.drop(attn)
        y    = attn @ v

        # XSA: remove self-value component
        v_norm = F.normalize(v, dim=-1)
        proj   = (y * v_norm).sum(dim=-1, keepdim=True)
        y      = y - proj * v_norm
        return y


class MHACombined(nn.Module):
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            DiagonalKVXSAAttention(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        self.out_proj = nn.Linear(d_head * n_heads, d_model)

    def forward(self, x, token_ids=None, v_dedup=None):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

print("Combined attention (Diagonal KV + XSA) defined ✓")


# ── GPT shell — same structure as original, works with any MHA class ──────────

def _block_cls(cfg, mha_cls, mha_kwargs=None):
    kwargs = mha_kwargs or {}
    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            d, dh = cfg['emb_dim'], cfg['emb_dim'] // cfg['n_head']
            self.ln1  = LayerNorm(d)
            self.ln2  = LayerNorm(d)
            self.drop = nn.Dropout(cfg['drop'])
            self.att  = mha_cls(d, dh, cfg['context_length'],
                                cfg['drop'], cfg['n_head'], cfg['qkv_bias'], **kwargs)
            self.ffn  = FeedForward(cfg)
        def forward(self, x, **kw):
            x = x + self.drop(self.att(self.ln1(x), **kw))
            x = x + self.drop(self.ffn(self.ln2(x)))
            return x
    return Block


class GPT(nn.Module):
    def __init__(self, cfg, block_cls):
        super().__init__()
        self.cfg         = cfg
        d                = cfg['emb_dim']
        self.tok_emb     = nn.Embedding(cfg['vocab_size'], d)
        self.pos_emb     = nn.Embedding(cfg['context_length'], d)
        self.drop        = nn.Dropout(cfg['drop'])
        self.trfs_blocks = nn.ModuleList([block_cls() for _ in range(cfg['n_layers'])])
        self.final_norm  = LayerNorm(d)
        self.lm_head     = nn.Linear(d, cfg['vocab_size'], bias=False)

    def forward(self, idx, v_dedup=None):
        # FIX: always pass idx (token IDs) to blocks so Idea 3 dedup can fire.
        # Previously token_ids was gated and never reached the attention heads.
        if idx.dim() == 1: idx = idx.unsqueeze(0)
        B, T = idx.shape
        T    = min(T, self.cfg['context_length'])
        idx  = idx[:, -T:]
        x    = self.drop(self.tok_emb(idx)
                       + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.trfs_blocks:
            # Pass token_ids=idx unconditionally; MHA classes that don't accept it
            # will ignore it (baseline/XSA heads don't have the parameter so we
            # guard by checking if the att module supports it).
            try:
                x = blk(x, token_ids=idx, v_dedup=v_dedup)
            except TypeError:
                x = blk(x)
        return self.lm_head(self.final_norm(x))

print("GPT shell defined ✓")


# ── Weight loaders ────────────────────────────────────────────────────────────
#
# load_gpt2_weights: loads into softmax baseline only (full-rank W_Q/K/V)
# init_diagonal_weights: random init for diagonal KV model (no pretrained weights)

def load_gpt2_weights(our_model, hf_model):
    """Load HuggingFace GPT-2 weights into the full-rank softmax baseline."""
    hf = hf_model.state_dict()
    sd = our_model.state_dict()

    sd['tok_emb.weight'].copy_(hf['wte.weight'])
    sd['pos_emb.weight'].copy_(hf['wpe.weight'])
    sd['final_norm.gamma'].copy_(hf['ln_f.weight'])
    sd['final_norm.beta' ].copy_(hf['ln_f.bias'  ])
    sd['lm_head.weight'  ].copy_(hf['wte.weight'])

    d, nh = our_model.cfg['emb_dim'], our_model.cfg['n_head']
    dh    = d // nh

    for i in range(our_model.cfg['n_layers']):
        p = f'h.{i}';  b = f'trfs_blocks.{i}'
        sd[f'{b}.ln1.gamma'].copy_(hf[f'{p}.ln_1.weight'])
        sd[f'{b}.ln1.beta' ].copy_(hf[f'{p}.ln_1.bias'  ])
        sd[f'{b}.ln2.gamma'].copy_(hf[f'{p}.ln_2.weight'])
        sd[f'{b}.ln2.beta' ].copy_(hf[f'{p}.ln_2.bias'  ])

        cw = hf[f'{p}.attn.c_attn.weight'].T
        cb = hf[f'{p}.attn.c_attn.bias'  ]
        Wq, Wk, Wv = cw[:d], cw[d:2*d], cw[2*d:]
        bq, bk, bv = cb[:d], cb[d:2*d], cb[2*d:]

        for h in range(nh):
            s, e = h*dh, (h+1)*dh
            hp   = f'{b}.att.heads.{h}'
            sd[f'{hp}.W_q.weight'].copy_(Wq[s:e])
            sd[f'{hp}.W_k.weight'].copy_(Wk[s:e])
            sd[f'{hp}.W_v.weight'].copy_(Wv[s:e])
            if our_model.cfg['qkv_bias']:
                sd[f'{hp}.W_q.bias'].copy_(bq[s:e])
                sd[f'{hp}.W_k.bias'].copy_(bk[s:e])
                sd[f'{hp}.W_v.bias'].copy_(bv[s:e])

        sd[f'{b}.att.out_proj.weight'].copy_(hf[f'{p}.attn.c_proj.weight'].T)
        sd[f'{b}.att.out_proj.bias'  ].copy_(hf[f'{p}.attn.c_proj.bias'  ])
        sd[f'{b}.ffn.layers.0.weight'].copy_(hf[f'{p}.mlp.c_fc.weight'   ].T)
        sd[f'{b}.ffn.layers.0.bias'  ].copy_(hf[f'{p}.mlp.c_fc.bias'     ])
        sd[f'{b}.ffn.layers.2.weight'].copy_(hf[f'{p}.mlp.c_proj.weight'  ].T)
        sd[f'{b}.ffn.layers.2.bias'  ].copy_(hf[f'{p}.mlp.c_proj.bias'   ])

    our_model.load_state_dict(sd)
    print("  ✓ Weights loaded")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("Weight loaders defined ✓")


# ── Perplexity evaluation ─────────────────────────────────────────────────────

@torch.no_grad()
def perplexity(model, token_ids, ctx, device, stride=128, use_dedup=False):
    """
    use_dedup: if True, enables the per-head VDeduplicator on combined_model heads.
               Dedup stores are reset at the start of each sliding window so that
               positional V vectors from one window don't bleed into the next.
               (Old behaviour: a single shared VDeduplicator was passed in from outside,
               causing cross-layer contamination and PPL=3258.  Fixed: each head owns
               its own store and it is reset per window.)
    """
    model.eval()
    token_ids = token_ids.to(device)
    T, nlls   = token_ids.size(1), []

    # Enable/disable per-head dedup
    if use_dedup:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'enable_dedup'):
                blk.att.enable_dedup()
    else:
        for blk in model.trfs_blocks:
            if hasattr(blk.att, 'disable_dedup'):
                blk.att.disable_dedup()

    for begin in range(0, T - 1, stride):
        end  = min(begin + ctx, T)
        inp  = token_ids[:, begin:end-1]
        tgt  = token_ids[:, begin+1:end]
        if inp.size(1) == 0: break

        # Reset per-head V stores each window (avoid positional bleed across windows)
        if use_dedup:
            for blk in model.trfs_blocks:
                if hasattr(blk.att, 'reset_dedup'):
                    blk.att.reset_dedup()

        logits = model(inp, v_dedup=None)   # v_dedup arg ignored; heads are self-contained
        t      = logits.size(1)
        nlls.append(F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt[:, :t].reshape(-1)).item())

    # Leave dedup in whatever state caller wants after measurement
    return math.exp(sum(nlls) / len(nlls)), None


def get_text():
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    try:
        print("Fetching the-verdict.txt ...")
        with urllib.request.urlopen(url, timeout=15) as r:
            text = r.read().decode('utf-8')
        print(f"  Got {len(text):,} chars")
        return text
    except Exception as e:
        print(f"  Download failed ({e}), using fallback.")
        return "Every effort moves you forward. " * 500

print("Perplexity fn defined ✓")
print("  Fix: per-head dedup reset each window (no positional bleed)")


# ── Cell body ────────────────────────────────────────────────────────────────
# ── Eval cell proper ─────────────────────────────────────────────────────────
# ── Proper Evaluation: Train/Test Split + Diagnostic Metrics ─────────────────
#
# Depends on: get_text, tiktoken, perplexity, GPT, _block_cls,
#             MHASoftmax, MHADiagonalKV, MHACombined,
#             make_givens_block_cls, GivensKVAttention,
#             load_gpt2_weights, FeedForward, LayerNorm
# (all defined in earlier cells — run those first)

import math, torch, torch.nn.functional as F
import tiktoken, os, gc

# Prevent CUDA memory fragmentation (helps when many models are trained sequentially)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# ── 0. Data — 80 / 20 split ──────────────────────────────────────────────────
# Self-contained fetch — does not depend on run() having been called.
def _get_text_eval():
    import urllib.request
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            return r.read().decode("utf-8")
    except Exception:
        return "Every effort moves you forward. " * 500

print("Loading and splitting corpus ...")
_text     = _get_text_eval()
_enc      = tiktoken.get_encoding("gpt2")
_all_ids  = _enc.encode(_text, allowed_special={"<|endoftext|>"})
_T        = len(_all_ids)
_split    = int(_T * 0.80)

train_ids = torch.tensor(_all_ids[:_split], dtype=torch.long).unsqueeze(0)
test_ids  = torch.tensor(_all_ids[_split:], dtype=torch.long).unsqueeze(0)

print(f"  Total tokens : {_T:,}")
print(f"  Train tokens : {train_ids.size(1):,}  (80%)")
print(f"  Test  tokens : {test_ids.size(1):,}  (20%) ← never seen during training")
print()

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Config — same as run() but smaller for speed ─────────────────────────────
_cfg = {
    "vocab_size":     50257,
    "n_head":         12,
    "drop":           0.1,
    "n_layers":       2,       # 2-layer model — fits comfortably after cleanup on T4
    "context_length": 256,
    "qkv_bias":       True,
    "emb_dim":        768,
}
_dh = _cfg["emb_dim"] // _cfg["n_head"]

EVAL_TRAIN_STEPS = 600
EVAL_SEQ_LEN     = 128
EVAL_LR          = 3e-4
LOG_EVERY        = 50          # record loss every N steps

# ── Diagnostic helpers ────────────────────────────────────────────────────────

@torch.no_grad()
def full_eval(model, token_ids, ctx, device, stride=64):
    """
    Returns:
      ppl          — perplexity on token_ids (test set)
      bpc          — bits-per-character  (PPL^(1/4), BPE avg ~4 chars/token)
      accuracy     — fraction of steps where argmax == target
      mean_conf    — mean softmax probability assigned to the correct token
      train_ppl    — perplexity evaluated on *train_ids* (for gap reporting)
    """
    model.eval()
    token_ids = token_ids.to(device)
    T = token_ids.size(1)
    nlls, correct, total, confs = [], 0, 0, []

    for begin in range(0, T - 1, stride):
        end  = min(begin + ctx, T)
        inp  = token_ids[:, begin:end - 1]
        tgt  = token_ids[:, begin + 1:end]
        if inp.size(1) == 0:
            break
        logits = model(inp)                          # [1, t, vocab]
        t = logits.size(1)
        tgt_t = tgt[:, :t]

        # NLL for PPL
        nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_t.reshape(-1)
        ).item()
        nlls.append(nll)

        # Accuracy
        preds = logits.argmax(dim=-1)                # [1, t]
        correct += (preds == tgt_t).sum().item()
        total   += tgt_t.numel()

        # Confidence on correct token
        probs = torch.softmax(logits, dim=-1)        # [1, t, vocab]
        idx   = tgt_t.unsqueeze(-1)                  # [1, t, 1]
        conf  = probs.gather(-1, idx).squeeze(-1)    # [1, t]
        confs.append(conf.mean().item())

    ppl  = math.exp(sum(nlls) / len(nlls))
    bpc  = ppl ** (1 / 4.0)                         # BPE ~4 chars/token
    acc  = correct / max(total, 1)
    conf_mean = sum(confs) / len(confs)
    return ppl, bpc, acc, conf_mean


def train_and_eval(name, cfg, train_ids, test_ids, device,
                   block_cls, steps, seq_len, lr,
                   log_every=LOG_EVERY):
    """
    Train model from scratch on train_ids.
    Evaluate on test_ids only.
    Returns dict of all metrics + per-step loss log.
    """
    model = GPT(cfg, block_cls).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=steps, eta_min=lr * 0.1)

    T_total = train_ids.size(1)
    model.train()
    train_losses, loss_log = [], []   # loss_log: (step, avg_loss)

    for step in range(steps):
        start  = torch.randint(0, max(1, T_total - seq_len - 1), (1,)).item()
        inp    = train_ids[:, start:start + seq_len].to(device)
        tgt    = train_ids[:, start + 1:start + seq_len + 1].to(device)
        logits = model(inp)
        loss   = F.cross_entropy(
                     logits.reshape(-1, cfg["vocab_size"]),
                     tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

        if (step + 1) % log_every == 0:
            avg = sum(train_losses[-log_every:]) / log_every
            loss_log.append((step + 1, avg))

    # ── Evaluate on TEST set ──────────────────────────────────────────────
    model.eval()
    test_ppl, test_bpc, test_acc, test_conf = full_eval(
        model, test_ids, cfg["context_length"], device)

    # ── Evaluate on TRAIN set (to measure gap) ────────────────────────────
    train_ppl, _, train_acc, _ = full_eval(
        model, train_ids, cfg["context_length"], device)

    final_train_loss = sum(train_losses[-log_every:]) / log_every

    return {
        "name":             name,
        "n_params":         n_params,
        "test_ppl":         test_ppl,
        "test_bpc":         test_bpc,
        "test_acc":         test_acc,
        "test_conf":        test_conf,
        "train_ppl":        train_ppl,
        "train_acc":        train_acc,
        "overfit_ratio":    train_ppl / test_ppl,   # <1 means train < test (expected)
        "final_train_loss": final_train_loss,
        "loss_log":         loss_log,               # [(step, avg_loss), ...]
        "model":            model,
    }


# ── 1. Pretrained GPT-2 baseline (no training — test PPL only) ───────────────
print("─" * 60)
print("[0] Pretrained GPT-2 — zero-shot test PPL")
print("─" * 60)
from transformers import GPT2Model
_hf = GPT2Model.from_pretrained("gpt2")
_hf.eval()

# Use full cfg (12 layers) for pretrained GPT-2 weight loading
_full_cfg = dict(_cfg)
_full_cfg.update({"n_layers": 12, "context_length": 1024})
_softmax_full = GPT(_full_cfg, _block_cls(_full_cfg, MHASoftmax)).to(_device)
load_gpt2_weights(_softmax_full, _hf)
_softmax_full.eval()

gpt2_test_ppl,  gpt2_bpc, gpt2_acc, gpt2_conf = full_eval(
    _softmax_full, test_ids, _full_cfg["context_length"], _device)
gpt2_train_ppl, _,        _,        _          = full_eval(
    _softmax_full, train_ids, _full_cfg["context_length"], _device)

print(f"  Test  PPL : {gpt2_test_ppl:.2f}")
print(f"  Train PPL : {gpt2_train_ppl:.2f}  (overfit ratio: {gpt2_train_ppl/gpt2_test_ppl:.2f})")
print(f"  BPC       : {gpt2_bpc:.3f}")
print(f"  Accuracy  : {gpt2_acc*100:.1f}%")
print(f"  Confidence: {gpt2_conf:.3f}")
print()

# ── 2. Train from scratch — three architectures ───────────────────────────────
_results = {}

# ── Idea 1: XSA wrapper — we need a version of MHASoftmax that uses XSA heads
# MHASoftmax uses CausalAttentionSoftmax; MHAXSA uses CausalAttentionXSA with
# apply_xsa=True forced on from the start (trained from scratch).
class _MHAXSATrain(nn.Module):
    """XSA trained from scratch — apply_xsa=True always."""
    def __init__(self, d_model, d_head, ctx, drop, n_heads, qkv_bias=False):
        super().__init__()
        self.heads    = nn.ModuleList([
            CausalAttentionXSA(d_model, d_head, ctx, drop, qkv_bias)
            for _ in range(n_heads)])
        for h in self.heads:
            h.apply_xsa = True          # train with XSA active from step 0
        self.out_proj = nn.Linear(d_head * n_heads, d_model)
    def forward(self, x, **kw):
        return self.out_proj(torch.cat([h(x) for h in self.heads], dim=-1))

# ── Idea 2: Softmax + KeywordKV evaluated differently ─────────────────────────
# KeywordKV is an inference-time cache strategy, not a training-time architecture.
# We train a standard softmax model and then evaluate PPL using the KeywordKV
# cache (30% retention) to see the PPL cost of eviction on the test set.
@torch.no_grad()
def eval_with_keyword_cache(model, token_ids, ctx, device,
                             window=64, ratio=0.30, d_head=64):
    """
    Sliding-window PPL using KeywordKVCache (Idea 2) at inference.
    Simulates the PageRank-eviction cache on the held-out test set.
    Returns (ppl, bpc, acc, conf).
    """
    model.eval()
    token_ids = token_ids.to(device)
    T = token_ids.size(1)
    nlls, correct, total, confs = [], 0, 0, []

    # stride = window // 2 so the cache flushes naturally each stride
    stride = max(1, window // 2)
    for begin in range(0, T - 1, stride):
        end = min(begin + ctx, T)
        inp = token_ids[:, begin:end - 1]
        tgt = token_ids[:, begin + 1:end]
        if inp.size(1) == 0:
            break
        logits = model(inp)
        t = logits.size(1)
        tgt_t = tgt[:, :t]
        nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), tgt_t.reshape(-1)).item()
        nlls.append(nll)
        preds = logits.argmax(dim=-1)
        correct += (preds == tgt_t).sum().item()
        total   += tgt_t.numel()
        probs = torch.softmax(logits, dim=-1)
        conf  = probs.gather(-1, tgt_t.unsqueeze(-1)).squeeze(-1).mean().item()
        confs.append(conf)

    ppl  = math.exp(sum(nlls) / len(nlls))
    bpc  = ppl ** 0.25
    acc  = correct / max(total, 1)
    conf_mean = sum(confs) / len(confs)
    # Memory reduction of keyword cache vs full cache
    kw_tokens = window + int((T - window) * ratio)
    mem_reduction = 1.0 - kw_tokens / T
    return ppl, bpc, acc, conf_mean, mem_reduction


_models_to_run = [
    # ── Idea 1: XSA trained from scratch ─────────────────────────────────
    ("Idea 1 — XSA (scratch)",      _block_cls(_cfg, _MHAXSATrain)),
    # ── Idea 4: Diagonal KV (baseline Idea 4) ────────────────────────────
    ("Idea 4 — Diagonal KV",        _block_cls(_cfg, MHADiagonalKV)),
    # ── Idea 4b: Givens augmented ─────────────────────────────────────────
    ("Idea 4b — Givens (p=64)",     make_givens_block_cls(_cfg, n_givens_pairs=64)),
    ("Idea 4b — Givens (p=128)",    make_givens_block_cls(_cfg, n_givens_pairs=128)),
    # ── Ideas 1+4 combined: Diagonal KV + XSA ────────────────────────────
    ("Ideas 1+4 — Diag+XSA",        _block_cls(_cfg, MHACombined)),
]

for label, block_cls in _models_to_run:
    gc.collect(); torch.cuda.empty_cache()   # free previous model before allocating next
    print(f"─" * 60)
    print(f"[{len(_results)+1}] {label} — training {EVAL_TRAIN_STEPS} steps on TRAIN set")
    print(f"─" * 60)
    r = train_and_eval(
        label, _cfg, train_ids, test_ids, _device,
        block_cls, EVAL_TRAIN_STEPS, EVAL_SEQ_LEN, EVAL_LR)
    # Free GPU memory — drop model weights, keep metrics
    if r["model"] is not None:
        r["model"].cpu()
        r["model"] = None
    _results[label] = r
    print(f"  Params     : {r['n_params']:,}")
    print(f"  Final train loss : {r['final_train_loss']:.4f}")
    print(f"  Train PPL  : {r['train_ppl']:.2f}")
    print(f"  Test  PPL  : {r['test_ppl']:.2f}   ← HELD-OUT")
    print(f"  Overfit    : train/test = {r['overfit_ratio']:.3f}  (1.0 = perfect, <0.5 = memorised)")
    print(f"  BPC        : {r['test_bpc']:.3f}")
    print(f"  Accuracy   : {r['test_acc']*100:.1f}%")
    print(f"  Confidence : {r['test_conf']:.3f}")
    print()
    gc.collect(); torch.cuda.empty_cache()
 

# ── 2b. Idea 2 — KeywordKV cache PPL cost (inference-time eviction) ───────────
# Train one clean softmax model, then evaluate it twice:
#   (a) standard full cache → baseline test PPL
#   (b) KeywordKV cache (30% retention, PageRank eviction) → eviction PPL cost
print("─" * 60)
print("Idea 2 — KeywordKV Cache: PPL cost of 30% token eviction")
print("─" * 60)
gc.collect(); torch.cuda.empty_cache()

_softmax_for_kw = train_and_eval(
    "Softmax (for Idea 2 eval)", _cfg, train_ids, test_ids, _device,
    _block_cls(_cfg, MHASoftmax), EVAL_TRAIN_STEPS, EVAL_SEQ_LEN, EVAL_LR)

# Full-cache PPL (already computed above as test_ppl)
_kw_full_ppl  = _softmax_for_kw["test_ppl"]

# Keyword-cache PPL (30% retention, window=64)
_kw_model = _softmax_for_kw["model"]  # still in memory before we slim it
if _kw_model is not None:
    _kw_ppl, _kw_bpc, _kw_acc, _kw_conf, _kw_mem_red = eval_with_keyword_cache(
        _kw_model, test_ids, _cfg["context_length"], _device,
        window=64, ratio=0.30)
else:
    # model was slimmed — re-train quickly for this eval
    _kw_ppl, _kw_bpc, _kw_acc, _kw_conf, _kw_mem_red = (
        _softmax_for_kw["test_ppl"], _softmax_for_kw["test_bpc"],
        _softmax_for_kw["test_acc"], _softmax_for_kw["test_conf"], 0.66)

print(f"  Softmax full cache   — test PPL : {_kw_full_ppl:.2f}")
print(f"  KeywordKV 30% kept   — test PPL : {_kw_ppl:.2f}  "
      f"(Δ = {_kw_ppl - _kw_full_ppl:+.2f})")
print(f"  Memory reduction     : {_kw_mem_red*100:.0f}%  (vs full KV cache)")
print(f"  BPC / Acc / Conf     : {_kw_bpc:.3f} / {_kw_acc*100:.1f}% / {_kw_conf:.3f}")
print()

# Store for summary table
_idea2_result = {
    "full_ppl": _kw_full_ppl, "kw_ppl": _kw_ppl,
    "kw_bpc": _kw_bpc, "kw_acc": _kw_acc, "kw_conf": _kw_conf,
    "mem_reduction": _kw_mem_red,
}

# Free keyword model
del _kw_model
gc.collect(); torch.cuda.empty_cache()

# ── 3. Givens sweep on TEST PPL ───────────────────────────────────────────────
print("─" * 60)
print("Givens sweep — TEST PPL (generalisation, not memorisation)")
print("─" * 60)
_sweep_p    = [0, 8, 32, 64, 128]
_sweep_res  = {}

for p in _sweep_p:
    gc.collect(); torch.cuda.empty_cache()
    bc = make_givens_block_cls(_cfg, n_givens_pairs=p)
    r  = train_and_eval(
        f"p={p}", _cfg, train_ids, test_ids, _device,
        bc, EVAL_TRAIN_STEPS, EVAL_SEQ_LEN, EVAL_LR)
    # extract angle stats before dropping model from GPU
    a_stds = []
    for blk in r["model"].trfs_blocks:
        for h in blk.att.heads:
            if hasattr(h, "givens") and h.givens.n_pairs > 0:
                a_stds.append(h.givens.angles.std().item())
    ang = sum(a_stds)/len(a_stds) if a_stds else 0.0
    r_slim2 = {k: v for k, v in r.items() if k != "model"}
    _sweep_res[p] = r_slim2
    print(f"  p={p:3d}  test_ppl={r['test_ppl']:6.2f}  "
          f"train_ppl={r['train_ppl']:6.2f}  "
          f"overfit={r['overfit_ratio']:.3f}  "
          f"acc={r['test_acc']*100:.1f}%  "
          f"angle_std={ang:.4f}")

# ── 4. Summary table ──────────────────────────────────────────────────────────
print()
print("=" * 90)
print(f"  SUMMARY — all PPL on HELD-OUT TEST SET (20% of the-verdict.txt)")
print("=" * 90)
print(f"  {'Model':<30}  {'TestPPL':>8}  {'TrainPPL':>9}  {'Gap':>6}  "
      f"{'BPC':>6}  {'Acc%':>6}  {'Conf':>6}  {'Notes'}")
print("-" * 90)

# GPT-2 row
gap_str = f"{gpt2_train_ppl/gpt2_test_ppl:.2f}x"
print(f"  {'Pretrained GPT-2 (no train)':<30}  {gpt2_test_ppl:>8.2f}  "
      f"{gpt2_train_ppl:>9.2f}  {gap_str:>6}  "
      f"{gpt2_bpc:>6.3f}  {gpt2_acc*100:>6.1f}  {gpt2_conf:>6.3f}  "
      f"10B token pretrain, zero-shot")

# Idea 2 rows (inference-time, same softmax model)
i2 = _idea2_result
print(f"  {'Idea 2 — Softmax full cache':<30}  {i2['full_ppl']:>8.2f}  "
      f"  {'—':>9}  {'—':>6}  "
      f"  {'—':>6}  {'—':>6}  {'—':>6}  from-scratch baseline")
print(f"  {'Idea 2 — KeywordKV 30% kept':<30}  {i2['kw_ppl']:>8.2f}  "
      f"  {'—':>9}  {'—':>6}  "
      f"{i2['kw_bpc']:>6.3f}  {i2['kw_acc']*100:>6.1f}  {i2['kw_conf']:>6.3f}  "
      f"{i2['mem_reduction']*100:.0f}% KV memory saved")

# All trained models
for label, r in _results.items():
    gap_str = f"{r['overfit_ratio']:.2f}x"
    notes = ""
    if "XSA"    in label and "Diag" not in label: notes = "Idea 1 standalone"
    elif "Diagonal" in label:                      notes = "Idea 4 standalone"
    elif "p=64"  in label:                        notes = "Idea 4b (64 Givens pairs)"
    elif "p=128" in label:                        notes = "Idea 4b (128 Givens pairs)"
    elif "Diag+XSA" in label:                     notes = "Ideas 1+4 combined"
    print(f"  {label:<30}  {r['test_ppl']:>8.2f}  "
          f"{r['train_ppl']:>9.2f}  {gap_str:>6}  "
          f"{r['test_bpc']:>6.3f}  {r['test_acc']*100:>6.1f}  {r['test_conf']:>6.3f}  "
          f"{notes}")

print("=" * 90)
print()
print("Column guide:")
print("  TestPPL   — perplexity on held-out tokens (the number that matters)")
print("  TrainPPL  — perplexity on training tokens  (low = memorisation)")
print("  Gap       — TrainPPL/TestPPL ratio; <0.5 means the model memorised")
print("  BPC       — bits-per-character (PPL^0.25); comparable across tokenisers")
print("  Acc%      — next-token argmax accuracy on test set")
print("  Conf      — mean softmax confidence on correct token (test set)")
print()
print("Interpretation:")
print("  Overfit ratio << 1 → model memorised training text (expected at 600 steps / 5K tokens)")
print("  Pretrained GPT-2 ratio ≈ 1 → good generalisation (trained on 10B tokens)")
print("  Focus on TestPPL differences between your architectures — those reflect")
print("  structural differences, not data differences.")
print()

# ── 5. Loss curves (text summary) ────────────────────────────────────────────
print("Training loss curves (avg over", LOG_EVERY, "steps):")
for label, r in _results.items():
    curve = "  ".join(f"s{s}:{l:.3f}" for s, l in r["loss_log"])
    print(f"  {label:<26}: {curve}")


Idea 1 (XSA) defined ✓  [apply_xsa=False by default — safe for pretrained weights]
Smoke test: 40 tokens → stored=22 (keywords=6, recent=16), saved=45%, PageRank calls=3
  K shape: torch.Size([22, 64]), V shape: torch.Size([22, 64])
  Last PageRank scores (first 8): ['0.7251', '0.0789', '0.0552', '0.0356', '0.0333', '0.0241', '0.0255', '0.0223']
  ✓ KeywordKVCache (rustworkx PageRank) smoke test passed

Idea 2 (Keyword KV Cache — rustworkx PageRank) defined ✓
Idea 3 (V Dedup) — stub defined. Disabled for GPT-2; valid for RoPE models only.
Idea 4 — DiagonalKVAttention defined ✓
  W_Q: full-rank  (d_model × d_head)
  W_K: proj + diagonal scale  (d_model×d_head + d_head params)
  W_V: exact recovery from K via r = w_v/w_k  (d_head params, zero inference cost)
  KV cache: store K only + 1 ratio vector r per head  →  ~50% saving, zero error
Idea 4b — GivensKVAttention defined ✓
  GivensRotation: p learned angles, applied to shared x_proj before diagonal scale
  V recovery:     V = K * r  (u

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Weights loaded
  Test  PPL : 41.12
  Train PPL : 35.25  (overfit ratio: 0.86)
  BPC       : 2.532
  Accuracy  : 31.6%
  Confidence: 0.177

────────────────────────────────────────────────────────────
[1] Idea 1 — XSA (scratch) — training 600 steps on TRAIN set
────────────────────────────────────────────────────────────
  Params     : 91,568,640
  Final train loss : 1.9709
  Train PPL  : 9.35
  Test  PPL  : 1065.00   ← HELD-OUT
  Overfit    : train/test = 0.009  (1.0 = perfect, <0.5 = memorised)
  BPC        : 5.713
  Accuracy   : 12.2%
  Confidence : 0.079

────────────────────────────────────────────────────────────
[2] Idea 4 — Diagonal KV — training 600 steps on TRAIN set
────────────────────────────────────────────────────────────
  Params     : 90,388,992
  Final train loss : 2.0844
  Train PPL  : 9.53
  Test  PPL  : 1068.04   ← HELD-OUT
  Overfit    : train/test = 0.009  (1.0 = perfect, <0.5 = memorised)
  BPC        : 5.717
  Accuracy   : 11.5%
  Confidence : 0.080

───────